# Paper figures
Run all cells on CPU to regenerate the 17 analytical figures in the paper and seven in the online appendix. Figure 2 is the separately drawn workflow diagram. The nine README plots come first. Each plot is written once as PNG and PDF in `001_Code/analysis/figures/054/`. The matched full-parameter study uses one seed per scale; the main comparisons use three.

In [ ]:
from io import BytesIO
from pathlib import Path
import hashlib
import json

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.text import Text
from matplotlib.transforms import Bbox
import numpy as np
import pandas as pd

root = next(path for path in (Path.cwd(), *Path.cwd().parents)
            if (path / "001_Code/notebooks/054_paper_figures.ipynb").is_file())
CODE_ROOT = root / "001_Code"
RESULTS = CODE_ROOT / "results"
OUTPUT_DIR = CODE_ROOT / "analysis/figures/054"
with (CODE_ROOT / "analysis/figure_sources/054/JF01.json").open() as handle:
    settings = json.load(handle)
LAYOUT = settings["layout"]
REFERENCE_VALUES = settings["reference_values"]
source_hashes = settings["sources"]

In [ ]:
PALETTE = {
    "blue": "#3F6F9F",
    "purple": "#7A5AA6",
    "orange": "#D9822B",
    "green": "#3B8C75",
    "red": "#C75252",
    "grey": "#777777",
    "light_grey": "#D9D9D9",
}
SEED_COLOURS = {0: "#3F6F9F", 1: "#C75252", 2: "#3B8C75"}
SEED_HANDLES = [
    mpl.lines.Line2D([], [], marker="o", linestyle="none", color=colour,
                     markersize=5, label=f"Seed {seed}")
    for seed, colour in SEED_COLOURS.items()
]

mpl.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 300,
    "font.size": 10,
    "axes.titlesize": 11,
    "axes.labelsize": 10,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": False,
    "legend.frameon": False,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})

## Paired accuracy differences
Each pair uses the same 1,884 questions. The hard tier comprises high-school mathematics and formal logic. Its range spans the lowest and highest of the three observed seed differences.

In [ ]:
def read_predictions(path, hashes):
    expected_hash = hashes[path.relative_to(CODE_ROOT).as_posix()]
    with path.open("rb") as handle:
        data = handle.read()
    if hashlib.sha256(data).hexdigest() != expected_hash:
        raise ValueError(f"Saved predictions differ from the recorded input: {path}")
    rows = [json.loads(line) for line in data.splitlines()]
    if len(rows) != 1884:
        raise ValueError(f"{path}: expected 1,884 questions, found {len(rows)}")
    return rows


def accuracy(rows):
    return float(np.mean([bool(row["correct"]) for row in rows]))

In [ ]:
QWEN_PIPELINES = {
    "0.5B": {"params": .5, "tag": "006_COLAB_FULL_B05_sft_grpo_seed{seed}"},
    "1.5B": {"params": 1.5, "tag": "009_COLAB_FULL_B15_sft_grpo_seed{seed}"},
    "3B": {"params": 3.0, "tag": "003_COLAB_FULL_B3_sft_grpo_seed{seed}"},
    "7B": {"params": 7.0, "tag": "013_COLAB_FULL_B7_sft_grpo_seed{seed}"},
}


MCSFT_P8_TAGS = {
    "0.5B": "007_p8_COLAB_FULL_B05_sft_only_seed{seed}",
    "1.5B": "010_p8_COLAB_FULL_B15_sft_only_seed{seed}",
    "3B": "007_p8_COLAB_FULL_B3_sft_only_seed{seed}",
    "7B": "014_p8_COLAB_FULL_B7_sft_only_seed{seed}",
}
METHOD_TIER_SUBJECTS = {
    "easy": {"high_school_psychology", "marketing", "us_foreign_policy"},
    "medium": {"high_school_geography", "philosophy", "business_ethics"},
    "hard": {"high_school_mathematics", "formal_logic"},
}

def rebuilt_tier_accuracy(rows, tier):
    subset = [row for row in rows if row["subject"] in METHOD_TIER_SUBJECTS[tier]]
    assert subset
    return accuracy(subset)

method_rows = []
method_tier_rows = []
method_sources = []
for scale, spec in QWEN_PIPELINES.items():
    for seed in (0, 1, 2):
        grpo_path = RESULTS / spec["tag"].format(seed=seed) / "grpo_preds.jsonl"
        mc_path = RESULTS / MCSFT_P8_TAGS[scale].format(seed=seed) / "sft_preds.jsonl"
        grpo_rows = read_predictions(grpo_path, source_hashes)
        mc_rows = read_predictions(mc_path, source_hashes)
        grpo_counts = {subject: sum(row["subject"] == subject for row in grpo_rows)
                       for subject in set(row["subject"] for row in grpo_rows)}
        mc_counts = {subject: sum(row["subject"] == subject for row in mc_rows)
                     for subject in set(row["subject"] for row in mc_rows)}
        assert grpo_counts == mc_counts, f"{scale} seed {seed}: subject coverage changed"
        # Compare within subjects because older exports order the subject blocks differently.
        for subject in grpo_counts:
            grpo_gold = [r["gold"] for r in grpo_rows if r["subject"] == subject]
            mc_gold = [r["gold"] for r in mc_rows if r["subject"] == subject]
            assert grpo_gold == mc_gold, f"{scale} seed {seed}: answer sequence differs for {subject}"
        method_sources.extend((grpo_path, mc_path))
        grpo_acc = accuracy(grpo_rows)
        mc_acc = accuracy(mc_rows)
        method_rows.append({
            "scale": scale, "params": spec["params"], "seed": seed,
            "grpo": grpo_acc, "mc_p8": mc_acc, "grpo_minus_mc": grpo_acc - mc_acc,
        })
        for tier in ("easy", "medium", "hard"):
            method_tier_rows.append({
                "scale": scale, "params": spec["params"], "seed": seed, "tier": tier,
                "difference": rebuilt_tier_accuracy(grpo_rows, tier) -
                              rebuilt_tier_accuracy(mc_rows, tier),
            })
method_df = pd.DataFrame(method_rows)
method_tier_df = pd.DataFrame(method_tier_rows)


assert set(path.relative_to(CODE_ROOT).as_posix() for path in method_sources) == set(source_hashes)
pd.testing.assert_frame_equal(method_df, pd.DataFrame(REFERENCE_VALUES["overall"]), check_exact=True)
pd.testing.assert_frame_equal(method_tier_df, pd.DataFrame(REFERENCE_VALUES["tiers"]), check_exact=True)

## Figure 1. Warm GRPO versus Plain MC-SFT trained for eight passes
Panel A shows the mean difference across seeds. Panel B shows the individual hard-tier differences, with grey bars for the observed range and red diamonds for the mean.

In [ ]:
def finish_axes(ax, ylabel=None, xlabel=None, zero=False):
    if zero:
        ax.axhline(0, color="#555555", linewidth=1, zorder=1.75)
    if ylabel:
        ax.set_ylabel(ylabel)
    if xlabel:
        ax.set_xlabel(xlabel)
    ax.grid(axis="y", color="#E8E8E8", linewidth=.7)


def style_jf01(fig):
    fig.set_size_inches(*LAYOUT["figsize_inches"])
    for text in fig.findobj(Text):
        if text.get_text():
            text.set_fontsize(max(text.get_fontsize(), 12))
    for ax in fig.axes:
        ax.tick_params(labelsize=12)
        ax.xaxis.label.set_fontsize(13)
        ax.yaxis.label.set_fontsize(13)
        ax.title.set_fontsize(14)
        for text in ax.get_legend().get_texts():
            text.set_fontsize(12)
    fig.axes[0].set_title("Three-seed mean difference by\ndifficulty", fontsize=14, pad=12)
    fig.axes[1].set_title("Hard-tier seed values and observed\nrange", fontsize=14, pad=12)
    fig.tight_layout(pad=1.0, w_pad=2.4, h_pad=2)
    fig.axes[1].legend(handles=SEED_HANDLES, ncol=1, loc="lower right", fontsize=11,
                       labelspacing=.3, borderaxespad=.4)
    for letter, ax, position in zip("AB", fig.axes, LAYOUT["axes_positions"]):
        ax.set_position(position)
        ax.text(-.025, 1.05, letter, transform=ax.transAxes,
                ha="right", fontsize=12, fontweight="bold")
    fig.canvas.draw()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12.6, 4.9), sharey=True)
params = [item["params"] for item in QWEN_PIPELINES.values()]
tier_styles = {
    "easy": (PALETTE["green"], "o", 1.7),
    "medium": (PALETTE["blue"], "s", 1.7),
    "hard": (PALETTE["red"], "D", 2.7),
    "overall": (PALETTE["grey"], "^", 1.7),
}
for tier, (colour, marker, width) in tier_styles.items():
    means = []
    for scale in QWEN_PIPELINES:
        if tier == "overall":
            values = method_df.loc[method_df.scale == scale, "grpo_minus_mc"]
        else:
            values = method_tier_df[(method_tier_df.scale == scale) &
                                    (method_tier_df.tier == tier)].difference
        means.append(values.mean() * 100)
    style = "--" if tier == "overall" else "-"
    axes[0].plot(params, means, marker=marker, color=colour, linestyle=style,
                 linewidth=width, label=tier.capitalize())

for scale, spec in QWEN_PIPELINES.items():
    frame = method_tier_df[(method_tier_df.scale == scale) &
                           (method_tier_df.tier == "hard")].sort_values("seed")
    values = frame.difference.to_numpy() * 100
    axes[1].scatter(np.full(3, spec["params"]), values,
                    color=[SEED_COLOURS[int(seed)] for seed in frame.seed], s=42, zorder=3)
    mean, low, high = values.mean(), values.min(), values.max()
    marker, caps, stems = axes[1].errorbar(
        spec["params"], mean, yerr=[[mean - low], [high - mean]],
        fmt="D", color=PALETTE["red"], ecolor="#888888", capsize=4, markersize=6, zorder=4)
    # Keep the observed range above seed circles and the mean above the range.
    marker.set_zorder(4.1)
    for cap in caps:
        cap.set_zorder(3.5)
        cap.set_color("#888888")
        cap.set_markeredgecolor("#888888")
        cap.set_markeredgewidth(.8)
    for stem in stems:
        stem.set_zorder(3.5)
        stem.set_linewidth(.8)
    label_side = -1 if scale == "7B" else 1
    axes[1].annotate(f"{mean:+.1f}", (spec["params"], mean),
                     xytext=(8 * label_side, 0), textcoords="offset points",
                     ha="right" if label_side < 0 else "left", va="center", fontsize=8)

for ax in axes:
    ax.set_xscale("log")
    ax.set_xticks(params, list(QWEN_PIPELINES))
    finish_axes(ax, ylabel="Warm GRPO minus Plain MMLU MC-SFT (p8)\n(percentage points)",
                xlabel="Model scale", zero=True)
axes[0].set_title("Three-seed mean difference by difficulty")
axes[0].legend(ncol=2)
axes[1].set_title("Hard-tier seed values and observed range")
axes[1].legend(handles=SEED_HANDLES, ncol=3, loc="lower right", fontsize=8)
fig.tight_layout()


hard_means = [method_tier_df[(method_tier_df.scale == scale) &
                           (method_tier_df.tier == "hard")].difference.mean() * 100
              for scale in QWEN_PIPELINES]
axes[1].plot(params, hard_means, color=PALETTE["red"], linewidth=2.7, zorder=2.5)
style_jf01(fig)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
name = "JF01_grpo_plain_mmlu_mc_sft"
bbox = Bbox.from_bounds(*LAYOUT["bbox_inches"])
fig.savefig(OUTPUT_DIR / f"{name}.png", dpi=300, bbox_inches=bbox, facecolor="white",
            metadata={"Software": None})
fig.savefig(OUTPUT_DIR / f"{name}.pdf", bbox_inches=bbox, facecolor="white",
            metadata={"Creator": None, "Producer": None, "CreationDate": None, "ModDate": None})
plt.show()

## Figure 3. Gains over the supervised warm start
Panel A shows generated accuracy at the GSM8K-SFT and Warm GRPO endpoints. Panel B pairs the two endpoints within each seed. Bands and whiskers span the three observed seeds. The table also reports the answer-letter probe for the same runs.

In [ ]:
with (CODE_ROOT / "analysis/figure_sources/054/JF02.json").open() as handle:
    stage_settings = json.load(handle)
stage_hashes = stage_settings["sources"]

stage_rows = []
stage_inputs = set()
for scale, spec in QWEN_PIPELINES.items():
    for seed in (0, 1, 2):
        folder = RESULTS / spec["tag"].format(seed=seed)
        files = {"sft": "sft_preds.jsonl", "grpo": "grpo_preds.jsonl",
                 "sft_probe": "sft_logprob_preds.jsonl", "grpo_probe": "grpo_logprob_preds.jsonl"}
        row = {"scale": scale, "params": spec["params"], "seed": seed}
        for metric, filename in files.items():
            path = folder / filename
            predictions = read_predictions(path, stage_hashes)
            items = [(p["subject"], p["gold"]) for p in predictions]
            if metric == "sft":
                reference_items = items
            else:
                assert items == reference_items, f"{scale} seed {seed}: item order differs for {metric}"
            row[metric] = accuracy(predictions)
            if metric in ("grpo", "grpo_probe"):
                row["hard_" + metric] = accuracy([p for p in predictions if p["tier"] == "hard"])
            stage_inputs.add(path.relative_to(CODE_ROOT).as_posix())
        stage_rows.append(row)

assert stage_inputs == set(stage_hashes) and len(stage_inputs) == 48
qwen_df = pd.DataFrame(stage_rows).sort_values(["params", "seed"])
qwen_df["delta"] = qwen_df["grpo"] - qwen_df["sft"]
qwen_df["probe_delta"] = qwen_df["grpo_probe"] - qwen_df["sft_probe"]
display(qwen_df.round(4))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12.2, 4.6))
for metric, label, colour, marker in (
    ("sft", "GSM8K-SFT", PALETTE["blue"], "o"),
    ("grpo", "Warm GRPO", PALETTE["orange"], "s"),
):
    means, lows, highs, params = [], [], [], []
    for scale, spec in QWEN_PIPELINES.items():
        values = qwen_df.loc[qwen_df.scale == scale, metric].to_numpy() * 100
        mean, low, high = (values.mean(), values.min(), values.max())
        params.append(spec["params"])
        means.append(mean)
        lows.append(low)
        highs.append(high)
        axes[0].scatter(np.full(3, spec["params"]), values, color=colour,
                        alpha=.25, s=22, zorder=2)
    means = np.asarray(means)
    axes[0].plot(params, means, marker=marker, color=colour, linewidth=2, label=label)
    axes[0].fill_between(params, lows, highs, color=colour, alpha=.12, zorder=1.6)

stage_means = []
for scale, spec in QWEN_PIPELINES.items():
    frame = qwen_df[qwen_df.scale == scale]
    values = frame.delta.to_numpy() * 100
    axes[1].scatter(np.full(3, spec["params"]), values,
                    color=[SEED_COLOURS[seed] for seed in frame.seed], s=38, zorder=3)
    mean, low, high = (values.mean(), values.min(), values.max())
    stage_means.append(mean)
    marker, caps, stems = axes[1].errorbar(
        spec["params"], mean, yerr=[[mean - low], [high - mean]],
        fmt="D", color="#222222", ecolor="#888888", capsize=4, markersize=5, zorder=4)
    marker.set_zorder(4.1)
    for cap in caps:
        cap.set_zorder(3.5)
        cap.set_color("#888888")
        cap.set_markeredgecolor("#888888")
        cap.set_markeredgewidth(.8)
    for stem in stems:
        stem.set_zorder(3.5)
        stem.set_linewidth(.8)

for ax in axes:
    ax.set_xscale("log")
    ax.set_xticks([.5, 1.5, 3, 7], ["0.5B", "1.5B", "3B", "7B"])
    finish_axes(ax, xlabel="Model scale")
axes[0].set_ylabel("MMLU generated accuracy (%)")
axes[0].set_title("GSM8K-SFT and Warm GRPO endpoints")
axes[0].legend()
finish_axes(axes[1], ylabel="Warm GRPO minus GSM8K-SFT (percentage points)", zero=True)
axes[1].set_title("Seed-level GRPO change")
axes[1].legend(handles=SEED_HANDLES, ncol=3, loc="upper right", fontsize=8)
fig.tight_layout()

fig.set_size_inches(10.4, 5.1)
for text in fig.findobj(Text):
    if text.get_text():
        text.set_fontsize(max(text.get_fontsize(), 12))
for ax in axes:
    ax.tick_params(labelsize=12)
    ax.xaxis.label.set_fontsize(13)
    ax.yaxis.label.set_fontsize(13)
    ax.title.set_fontsize(14)
    for text in ax.get_legend().get_texts():
        text.set_fontsize(12)
fig.tight_layout(pad=1.0, w_pad=2.4, h_pad=2)
axes[1].legend(handles=axes[1].get_legend().legend_handles,
               labels=["Seed 0", "Seed 1", "Seed 2"], loc="upper right", fontsize=12)
for letter, ax in zip("AB", axes):
    ax.text(-.025, 1.05, letter, transform=ax.transAxes,
            ha="right", fontsize=12, fontweight="bold")
fig.canvas.draw()

# Connect the paired means after setting the axes so the line preserves the observed limits.
np.testing.assert_allclose(stage_means, stage_settings["mean_gain_points"], rtol=0, atol=1e-12)
axes[1].set_autoscale_on(False)
axes[1].plot(params, stage_means, color="#222222", linewidth=1.5, zorder=2.5)

name = "JF02_qwen_scale_response"
fig.savefig(OUTPUT_DIR / f"{name}.png", dpi=300, bbox_inches="tight", facecolor="white",
            metadata={"Software": None})
fig.savefig(OUTPUT_DIR / f"{name}.pdf", bbox_inches="tight", facecolor="white",
            metadata={"Creator": None, "Producer": None, "CreationDate": None, "ModDate": None})
plt.show()

## Qwen2.5 accuracy by subject and training route
Figures 5, 7 and 17 use the same test set of 1,884 questions. There is one pretrained baseline per model size. The other routes have three endpoints per size. Both multiple-choice SFT routes use eight passes over the MMLU training pool.

In [ ]:
SUBJECT_TIERS = {
    "easy": ["high_school_psychology", "marketing", "us_foreign_policy"],
    "medium": ["high_school_geography", "philosophy", "business_ethics"],
    "hard": ["high_school_mathematics", "formal_logic"],
}

SUBJECT_ORDER = sum(SUBJECT_TIERS.values(), [])

SUBJECT_LABELS = [
    "Psychology", "Marketing", "US foreign policy", "Geography",
    "Philosophy", "Business ethics", "Mathematics", "Formal logic",
]

BASE_GRPO_PIPELINES = {
    "0.5B": "005_COLAB_FULL_B05_base_grpo_seed{seed}",
    "1.5B": "012_COLAB_FULL_B15_base_grpo_seed{seed}",
    "3B": "004_COLAB_FULL_B3_base_grpo_seed{seed}",
    "7B": "013_COLAB_FULL_B7_base_grpo_seed{seed}",
}

BASE_EVAL_TAGS = {
    "0.5B": "005_COLAB_FULL_B05", "1.5B": "012_COLAB_FULL_B15",
    "3B": "004_COLAB_FULL_B3", "7B": "013_COLAB_FULL_B7",
}
WARM_MCSFT_TAGS = {
    "0.5B": {0: "008_wsft_COLAB_FULL_B05_sft_only_seed0",
              1: "032_wsft_COLAB_FULL_B05_sft_only_seed1",
              2: "032_wsft_COLAB_FULL_B05_sft_only_seed2"},
    "1.5B": {0: "011_wsft_COLAB_FULL_B15_sft_only_seed0",
              1: "032_wsft_COLAB_FULL_B15_sft_only_seed1",
              2: "032_wsft_COLAB_FULL_B15_sft_only_seed2"},
    "3B": {0: "008_wsft_COLAB_FULL_B3_sft_only_seed0",
            1: "032_wsft_COLAB_FULL_B3_sft_only_seed1",
            2: "032_wsft_COLAB_FULL_B3_sft_only_seed2"},
    "7B": {0: "015_wsft_COLAB_FULL_B7_sft_only_seed0",
            1: "032_wsft_COLAB_FULL_B7_sft_only_seed1",
            2: "032_wsft_COLAB_FULL_B7_sft_only_seed2"},
}
with (CODE_ROOT / "analysis/figure_sources/054/qwen_routes.json").open() as handle:
    route_hashes = json.load(handle)["sources"]

In [ ]:
def endpoint_accuracy(path, hashes):
    rows = read_predictions(path, hashes)
    assert all(isinstance(row["correct"], bool) for row in rows), path
    by_subject = {name: [] for name in SUBJECT_ORDER}
    gold = {name: [] for name in SUBJECT_ORDER}
    for row in rows:
        by_subject[row["subject"]].append(row["correct"])
        gold[row["subject"]].append(row["gold"])
    assert all(by_subject.values()), f"Missing subject in {path}"
    values = [accuracy(rows)] + [np.mean(by_subject[name]) for name in SUBJECT_ORDER]
    return np.array(values), gold

endpoints = {}
used_paths = set()
for scale, spec in QWEN_PIPELINES.items():
    base = RESULTS / "base_evals" / BASE_EVAL_TAGS[scale] / "base_preds.jsonl"
    endpoints[scale] = {}
    base_values, gold = endpoint_accuracy(base, route_hashes)
    endpoints[scale]["Pretrained base"] = base_values
    used_paths.add(base)
    for seed in (0, 1, 2):
        pipeline = RESULTS / spec["tag"].format(seed=seed)
        paths = {
            "Warm GRPO": pipeline / "grpo_preds.jsonl",
            "GSM8K-SFT": pipeline / "sft_preds.jsonl",
            "Cold GRPO": RESULTS / BASE_GRPO_PIPELINES[scale].format(seed=seed) / "grpo_preds.jsonl",
            "Plain MMLU MC-SFT": RESULTS / MCSFT_P8_TAGS[scale].format(seed=seed) / "sft_preds.jsonl",
            "Warm MMLU MC-SFT": RESULTS / WARM_MCSFT_TAGS[scale][seed] / "sft_preds.jsonl",
        }
        for route, path in paths.items():
            values, route_gold = endpoint_accuracy(path, route_hashes)
            assert route_gold == gold, f"{scale} {route} seed {seed}: subject answer sequence differs"
            endpoints[scale].setdefault(route, []).append(values)
            used_paths.add(path)
    for route in paths:
        endpoints[scale][route] = np.stack(endpoints[scale][route])

assert {p.relative_to(CODE_ROOT).as_posix() for p in used_paths} == set(route_hashes)

In [ ]:
def save_plot(fig, stem):
    fig.savefig(OUTPUT_DIR / f"{stem}.png", dpi=300, bbox_inches="tight", facecolor="white",
                metadata={"Software": None})
    fig.savefig(OUTPUT_DIR / f"{stem}.pdf", bbox_inches="tight", facecolor="white",
                metadata={"Creator": None, "Producer": None, "CreationDate": None, "ModDate": None})
    plt.show()


def size_single_panel(fig, ax, height):
    fig.set_size_inches(10.4, height)
    for text in fig.findobj(Text):
        if text.get_text():
            text.set_fontsize(max(text.get_fontsize(), 12))
    ax.tick_params(labelsize=12)
    ax.xaxis.label.set_fontsize(13)
    ax.yaxis.label.set_fontsize(13)
    ax.title.set_fontsize(14)
    for text in ax.get_legend().get_texts():
        text.set_fontsize(12)
    fig.tight_layout(pad=1.0, w_pad=2.4, h_pad=2)

## Figure 5. Warm GRPO accuracy by subject
Lines show the mean of three seeds and bands span their observed range. The dashed line marks the 25% accuracy expected from uniform choice among four answers.

In [ ]:
TIER_BACKGROUNDS = {"easy": "#EAF5EE", "medium": "#E8F0F8", "hard": "#FCEAEA"}
subject_scale_values = {scale: routes["Warm GRPO"][:, 1:] * 100 for scale, routes in endpoints.items()}
x = np.arange(len(SUBJECT_ORDER))
scale_colours = {
    "0.5B": PALETTE["blue"], "1.5B": PALETTE["green"],
    "3B": PALETTE["orange"], "7B": PALETTE["red"],
}
fig, ax = plt.subplots(figsize=(12.8, 5.6))
for scale, values in subject_scale_values.items():
    means = values.mean(axis=0)
    ax.plot(x, means, marker="o", linewidth=2, color=scale_colours[scale], label=scale)
    ax.fill_between(x, values.min(axis=0), values.max(axis=0),
                    color=scale_colours[scale], alpha=.09, zorder=1.6)
for left, right, label, colour in (
    (-.5, 2.5, "easy", TIER_BACKGROUNDS["easy"]),
    (2.5, 5.5, "medium", TIER_BACKGROUNDS["medium"]),
    (5.5, 7.5, "hard", TIER_BACKGROUNDS["hard"]),
):
    ax.axvspan(left, right, color=colour, alpha=.65, zorder=0)
    ax.text((left + right) / 2, .985, label, transform=ax.get_xaxis_transform(),
            ha="center", va="top", fontsize=9)
ax.axhline(25, color=PALETTE["grey"], linestyle="--", linewidth=1, label="25% uniform choice")
ax.set_xticks(x, SUBJECT_LABELS, rotation=24, ha="right")
finish_axes(ax, ylabel="Warm GRPO generated accuracy (%)", xlabel="MMLU subject")
ax.legend(ncol=5, loc="lower center", bbox_to_anchor=(.5, 1.01), borderaxespad=0)
fig.tight_layout()
size_single_panel(fig, ax, 6.1)
guide = next(line for line in ax.lines if line.get_label() == "25% uniform choice")
guide.set_zorder(1.55)
save_plot(fig, "JF18_qwen_subject_scaling")

## Figure 7. Paired differences between training routes
Panel A shows accuracy differences over the full test set. Panels B to D show the three comparisons within each subject. The cells describe observed differences across seeds.

In [ ]:
comparison_names = [
    "Warm GRPO minus GSM8K-SFT",
    "Cold GRPO minus pretrained base",
    "Warm GRPO minus Plain MMLU MC-SFT (p8)",
]
comparison_colours = [PALETTE["orange"], PALETTE["purple"], PALETTE["green"]]
route_pairs = [
    ("Warm GRPO", "GSM8K-SFT"),
    ("Cold GRPO", "Pretrained base"),
    ("Warm GRPO", "Plain MMLU MC-SFT"),
]
overall_route_values = {}
subject_route_values = {}
for label, (route, baseline) in zip(comparison_names, route_pairs):
    means = []
    for scale in QWEN_PIPELINES:
        paired = (endpoints[scale][route] - endpoints[scale][baseline]) * 100
        means.append(paired.mean(axis=0))
    differences = np.stack(means)
    overall_route_values[label] = differences[:, 0]
    subject_route_values[label] = differences[:, 1:].T

fig = plt.figure(figsize=(8.8, 8.3), layout="constrained")
grid = fig.add_gridspec(2, 3, height_ratios=[.78, 1.45])
top = fig.add_subplot(grid[0, :])
params = [spec["params"] for spec in QWEN_PIPELINES.values()]
label_offsets = {
    comparison_names[0]: ((0, -16), (0, 8), (0, 8), (-16, -16)),
    comparison_names[1]: ((0, 8), (0, 8), (0, 8), (-16, 10)),
    comparison_names[2]: ((0, 8), (0, -16), (0, 8), (18, 4)),
}
for name, colour, marker in zip(comparison_names, comparison_colours, ("o", "s", "^")):
    top.plot(params, overall_route_values[name], marker=marker, linewidth=2,
             color=colour, label=name)
    for x_value, value, offset in zip(params, overall_route_values[name], label_offsets[name]):
        top.annotate(f"{value:+.1f}", (x_value, value), xytext=offset,
                     textcoords="offset points", ha="center", fontsize=10)
top.set_xscale("log")
top.set_xticks(params, list(QWEN_PIPELINES))
top.set_ylim(top.get_ylim()[0], top.get_ylim()[1] + 4)
finish_axes(top, ylabel="Accuracy difference (percentage points)",
            xlabel="Model scale", zero=True)
top.set_title("Overall route differences across all 1,884 test questions")
top.legend(ncol=1, loc="lower left", bbox_to_anchor=(0, 1.15), borderaxespad=0, fontsize=10)

heat_limit = np.ceil(max(np.abs(values).max() for values in subject_route_values.values()))
heat_axes = []
for index, name in enumerate(comparison_names):
    ax = fig.add_subplot(grid[1, index], sharey=heat_axes[0] if heat_axes else None)
    values = subject_route_values[name]
    image = ax.imshow(values, cmap="RdYlGn", vmin=-heat_limit, vmax=heat_limit, aspect="auto")
    for row in range(values.shape[0]):
        for column in range(values.shape[1]):
            ax.text(column, row, f"{values[row, column]:+.1f}",
                    ha="center", va="center", fontsize=10)
    ax.set_xticks(range(4), list(QWEN_PIPELINES))
    ax.set_xlabel("Model scale")
    panel_titles = {
        comparison_names[0]: "Warm GRPO after\nGSM8K-SFT",
        comparison_names[1]: "Cold GRPO from the\npretrained base",
        comparison_names[2]: "Warm GRPO against\nPlain MMLU MC-SFT (p8)",
    }
    ax.set_title(panel_titles[name], fontsize=10)
    for boundary in (2.5, 5.5):
        ax.axhline(boundary, color="white", linewidth=2)
    if index == 0:
        ax.set_yticks(range(len(SUBJECT_LABELS)), SUBJECT_LABELS)
    else:
        ax.tick_params(labelleft=False)
    heat_axes.append(ax)
fig.colorbar(image, ax=heat_axes, shrink=.78, pad=.018, fraction=.025, aspect=28,
             label="Accuracy difference (percentage points)")
for letter, ax in zip("ABCD", [top, *heat_axes]):
    ax.text(-.025, 1.05, letter, transform=ax.transAxes, ha="right",
            fontsize=10.3, fontweight="bold")
# Settle the shared panel layout with both renderers before freezing it for export.
fig.savefig(BytesIO(), format="png", dpi=300, bbox_inches="tight")
fig.savefig(BytesIO(), format="pdf", bbox_inches="tight")
fig.set_layout_engine("none")
save_plot(fig, "JF19_qwen_route_synthesis_v40")

## Figure 17. Five training routes against the pretrained base
Each bar is the three-seed mean gain in generated accuracy over the base model at the same size. The routes used different amounts of computation.

In [ ]:
ROUTE_COLOURS = {
    "Pretrained base": PALETTE["grey"], "GSM8K-SFT": PALETTE["blue"],
    "Warm GRPO": PALETTE["orange"], "Cold GRPO": PALETTE["purple"],
    "Plain MMLU MC-SFT": PALETTE["green"], "Warm MMLU MC-SFT": PALETTE["red"],
}
route_order = ["Warm GRPO", "Cold GRPO", "GSM8K-SFT", "Plain MMLU MC-SFT", "Warm MMLU MC-SFT"]
route_matrix = np.array([
    [(endpoints[scale][route][:, 0].mean() - endpoints[scale]["Pretrained base"][0]) * 100
     for scale in QWEN_PIPELINES]
    for route in route_order
])

fig, ax = plt.subplots(figsize=(11.8, 5.5))
x = np.arange(len(QWEN_PIPELINES))
width = .18
for index, route in enumerate(route_order):
    values = route_matrix[index]
    bars = ax.bar(x + (index - 2) * width, values, width,
                  color=ROUTE_COLOURS[route], label=route + " (p8)" if "MC-SFT" in route else route)
    for bar, value in zip(bars, values):
        offset = .55 if value >= 0 else -.8
        ax.text(bar.get_x() + bar.get_width() / 2, value + offset, f"{value:+.1f}",
                ha="center", va="bottom" if value >= 0 else "top", fontsize=7.5)
ax.axhline(0, color="#888888", linewidth=.6, zorder=.75)
ax.set_axisbelow(True)
ax.set_xticks(x, list(QWEN_PIPELINES))
finish_axes(ax, ylabel="Gain over pretrained base (percentage points)",
            xlabel="Model scale")
ax.set_ylim(bottom=route_matrix.min() - 3.5)
ax.legend(ncol=2, loc="upper right")
fig.tight_layout()
size_single_panel(fig, ax, 6.2)
for text in ax.texts:
    text.set_fontsize(11)
ax.set_ylim(top=43)
handles, labels = ax.get_legend_handles_labels()
ax.get_legend().remove()
ax.legend(handles, labels, loc="upper right", ncol=1, fontsize=11.5)
fig.tight_layout(rect=(0, 0, 1, .84))
# Use the largest label size that leaves space between adjacent bars.
for size in np.arange(12, 7.75, -.25):
    for text in ax.texts:
        text.set_fontsize(size)
    fig.canvas.draw()
    boxes = [text.get_window_extent(fig.canvas.get_renderer()).expanded(1.06, 1.05)
             for text in ax.texts]
    if not any(a.overlaps(b) for i, a in enumerate(boxes) for b in boxes[i + 1:]):
        break
for tick in ax.yaxis.get_major_ticks():
    if abs(tick.get_loc()) < 1e-10:
        tick.gridline.set_visible(False)
save_plot(fig, "JF17_qwen_training_route_map_v40")

## Training rewards for Warm and Cold GRPO
The two figures show the accuracy and format rewards logged during training. The faint lines are raw means across three seeds. Each bold line averages the three trailing 25-update means, using shorter windows at the start. The shaded area spans the observed seed range.

In [ ]:
import yaml

WARM_REWARD_RUNS = {
    "Qwen2.5": {
        "0.5B": ("006_COLAB_FULL_B05_sft_grpo_seed{seed}", "006_colab_full_b05_seed{seed}.yaml"),
        "1.5B": ("009_COLAB_FULL_B15_sft_grpo_seed{seed}", "009_colab_full_b15_seed{seed}.yaml"),
        "3B": ("003_COLAB_FULL_B3_sft_grpo_seed{seed}", "003_colab_full_b3_seed{seed}.yaml"),
        "7B": ("013_COLAB_FULL_B7_sft_grpo_seed{seed}", "013_colab_full_b7_seed{seed}.yaml"),
    },
    "GPT-2": {
        "124M": ("048_COLAB_FULL_GPT2_M124_sft_grpo_seed{seed}", "048_colab_gpt2_m124_seed{seed}.yaml"),
        "355M": ("048_COLAB_FULL_GPT2_M355_sft_grpo_seed{seed}", "048_colab_gpt2_m355_seed{seed}.yaml"),
        "774M": ("048_COLAB_FULL_GPT2_M774_sft_grpo_seed{seed}", "048_colab_gpt2_m774_seed{seed}.yaml"),
        "XL 1.5B": ("046_COLAB_FULL_GPT2_B15_sft_grpo_seed{seed}", "046_colab_gpt2_b15_seed{seed}.yaml"),
    },
}

COLD_QWEN_RUNS = {
    "0.5B": ("005_COLAB_FULL_B05_base_grpo_seed{seed}", "005_colab_full_b05_base_grpo_seed{seed}.yaml"),
    "1.5B": ("012_COLAB_FULL_B15_base_grpo_seed{seed}", "012_colab_full_b15_base_grpo_seed{seed}.yaml"),
    "3B": ("004_COLAB_FULL_B3_base_grpo_seed{seed}", "004_colab_full_b3_base_grpo_seed{seed}.yaml"),
    "7B": ("013_COLAB_FULL_B7_base_grpo_seed{seed}", "013_colab_full_b7_base_grpo_seed{seed}.yaml"),
}

reward_source_dir = CODE_ROOT / "analysis/figure_sources/054"
with (reward_source_dir / "reward_curves.json").open() as handle:
    reward_sources = json.load(handle)
reward_hashes = reward_sources["sources"]
reward_inputs = set()


def reward_bytes(path):
    name = path.relative_to(CODE_ROOT).as_posix()
    data = path.read_bytes()
    if hashlib.sha256(data).hexdigest() != reward_hashes[name]:
        raise ValueError(f"Training record differs from the recorded input: {name}")
    reward_inputs.add(name)
    return data


def reward_config(path, seed, condition):
    cfg = yaml.safe_load(reward_bytes(path))
    assert cfg["condition"] == condition and cfg["train"]["seed"] == seed, path
    assert cfg["train"]["max_steps"] == 500 and cfg["train"]["num_generations"] == 8, path
    assert cfg["lora"]["r"] == 16 and cfg["grpo"]["beta"] == .04, path
    return cfg


def reward_frame(rows, family, scale, seed, steps, prefix=""):
    step_key = "train/global_step" if prefix else "step"
    frame = pd.DataFrame({
        "step": [r[step_key] for r in rows],
        "accuracy_reward": [r[prefix + "rewards/accuracy_reward/mean"] for r in rows],
        "format_reward": [r[prefix + "rewards/format_reward/mean"] for r in rows],
    })
    assert frame.step.tolist() == list(range(1, steps + 1)), (family, scale, seed)
    assert np.isfinite(frame.to_numpy()).all()
    assert frame.accuracy_reward.between(-1e-7, 1 + 1e-7).all()
    assert frame.format_reward.between(-1e-7, .2 + 1e-7).all()
    return frame.assign(family=family, scale=scale, seed=seed)


def completed_rewards(roster, condition):
    frames = []
    for family, sizes in roster.items():
        for scale, (folder, config) in sizes.items():
            for seed in (0, 1, 2):
                reward_config(CODE_ROOT / "configs" / config.format(seed=seed), seed, condition)
                path = RESULTS / folder.format(seed=seed) / "grpo_trainlog.json"
                log = json.loads(reward_bytes(path))
                rows = [r for r in log if "rewards/accuracy_reward/mean" in r]
                frames.append(reward_frame(rows, family, scale, seed, 500))
    return pd.concat(frames, ignore_index=True)

In [ ]:
warm_rewards = completed_rewards(WARM_REWARD_RUNS, "SFT_GRPO")
cold_qwen = completed_rewards({"Qwen2.5": COLD_QWEN_RUNS}, "BASE_GRPO")
cold_frames = [cold_qwen]
xl_records = reward_sources["cold_xl_runs"]
assert [r["seed"] for r in xl_records] == [0, 1, 2]

# Verify the stopped XL runs against their saved status and run identifiers.
for record in xl_records:
    seed, run_id = record["seed"], record["run_id"]
    folder = RESULTS / f"049_base_grpo_COLAB_FULL_GPT2_B15_base_grpo_seed{seed}"
    conclusion = json.loads(reward_bytes(folder / "049_a1_conclusion.json"))
    status_data = reward_bytes(folder / "grpo_worker_status.json")
    status = json.loads(status_data)
    binding_data = reward_bytes(folder / "049_grpo_wandb_runs" / f"{run_id}.json")
    binding = json.loads(binding_data)
    assert conclusion["completed"] and conclusion["outcome"] == "a1" and conclusion["seed"] == seed
    assert conclusion["status_sha256"] == hashlib.sha256(status_data).hexdigest()
    assert "sentinel abort: a1:" in status["error_message"] and status["cuda_oom"] is False
    assert len(conclusion["wandb_runs"]) == 1
    assert conclusion["wandb_runs"][0]["sha256"] == hashlib.sha256(binding_data).hexdigest()
    assert binding["run_id"] == run_id and binding["run_name"] == record["run_name"]
    assert binding["config_hash"] == conclusion["config_hash"]
    cfg = reward_config(CODE_ROOT / "configs" / f"049_base_grpo_colab_gpt2_b15_seed{seed}.yaml",
                        seed, "BASE_GRPO")
    assert cfg["grpo"]["max_completion_length"] == 512
    data = reward_bytes(reward_source_dir / record["file"])
    rows = [json.loads(line) for line in data.splitlines()]
    assert np.mean([r["train/completions/clipped_ratio"] for r in rows]) > .30
    cold_frames.append(reward_frame(rows, "GPT-2", "XL 1.5B", seed, 50, prefix="train/"))

cold_rewards = pd.concat(cold_frames, ignore_index=True)
assert len(warm_rewards) == 12000 and len(cold_rewards) == 6150
for records in (warm_rewards, cold_rewards):
    assert not records.duplicated(["family", "scale", "seed", "step"]).any()
assert reward_inputs == set(reward_hashes)

In [ ]:
def plot_rewards(records, stem, cold=False):
    scales = {family: list(sizes) for family, sizes in WARM_REWARD_RUNS.items()}
    if cold:
        scales["GPT-2"] = ["XL 1.5B"]
    colours = [PALETTE[k] for k in ("blue", "orange", "green", "red")]
    metrics = (("accuracy_reward", "Mean accuracy reward", 1.02),
               ("format_reward", "Mean format reward", .205))
    with mpl.rc_context({"font.size": 11, "axes.labelsize": 11, "axes.titlesize": 12}):
        fig, axes = plt.subplots(2, 2, figsize=(8.3, 5.0), sharex=not cold, sharey="row")
        for col, (family, sizes) in enumerate(scales.items()):
            stopped = cold and family == "GPT-2"
            steps = 50 if stopped else 500
            for row, (metric, label, upper) in enumerate(metrics):
                ax = axes[row, col]
                for scale, colour in zip(sizes, [PALETTE["red"]] if stopped else colours):
                    group = records[(records.family == family) & (records.scale == scale)]
                    raw = group.pivot(index="step", columns="seed", values=metric).reindex(columns=(0, 1, 2))
                    assert raw.shape == (steps, 3) and not raw.isna().any().any()
                    smooth = raw.rolling(25, min_periods=1).mean()
                    ax.plot(raw.index, raw.mean(axis=1), color=colour, alpha=.09, linewidth=.65)
                    ax.fill_between(raw.index, smooth.min(axis=1).to_numpy(),
                                    smooth.max(axis=1).to_numpy(), color=colour, alpha=.10, zorder=1.6)
                    ax.plot(raw.index, smooth.mean(axis=1).to_numpy(), color=colour,
                            linewidth=1.8, label=scale, zorder=2.5)
                ax.set_xlim(0, 52 if stopped else 500)
                ax.set_ylim(0, upper)
                finish_axes(ax, label if col == 0 else None, "GRPO update" if row == 1 else None)
                if cold:
                    ax.tick_params(axis="x", labelbottom=row == 1)
                if row == 0:
                    ax.set_title("GPT-2 (first 50 updates)" if stopped else family, pad=34)
                    ax.legend(ncol=len(sizes), loc="lower center", bbox_to_anchor=(.5, 1.01),
                              fontsize=9.5, handlelength=1.3, columnspacing=.8)
                if stopped:
                    ax.set_xticks(np.arange(0, 51, 10))
                    ax.axvline(50, color=PALETTE["grey"], linestyle=":", linewidth=1, zorder=1.55)
                    ax.text(.90, .88, "A1 stop at update 50\nAll three seeds; no endpoint",
                            transform=ax.transAxes, ha="right", va="top", fontsize=9.5, color="#555555")
        fig.tight_layout(h_pad=1.8, w_pad=1.8)
        for letter, ax in zip("ABCD", axes.flat):
            ax.text(-.025, 1.05, letter, transform=ax.transAxes, ha="right",
                    fontsize=9.5, fontweight="bold")
        save_plot(fig, stem)

## Figure 16. Warm GRPO training rewards
Qwen2.5 and GPT-2 each have four model sizes with three seeds per size. All 24 runs recorded 500 updates.

In [ ]:
plot_rewards(warm_rewards, "JF11_recorded_reward_trajectories")

## Figure S7. Cold GRPO training rewards
The twelve Qwen2.5 runs recorded 500 updates. All three GPT-2 XL runs stopped at update 50 and produced no evaluated endpoint. The smaller GPT-2 sizes were not run on this route.

In [ ]:
plot_rewards(cold_rewards, "JF12_cold_reward_trajectories", cold=True)

## Figure 8. Warm GRPO against the best tested MC-SFT alternative
At each size, compare the three-seed mean for each MC-SFT variant. Select the highest mean separately overall and for each of the eight subjects, retaining ties in the table. The variants are plain MC-SFT at 2, 8 and 32 passes and warm MC-SFT at 8 passes. At 7B both starts were also tested at 16 passes. Positive differences favour Warm GRPO.

In [ ]:
with (CODE_ROOT / "analysis/figure_sources/054/best_mc_sft.json").open() as handle:
    comparison_hashes = json.load(handle)["sources"]

plain_tags = {
    "0.5B": "007_p{passes}_COLAB_FULL_B05_sft_only_seed{seed}",
    "1.5B": "010_p{passes}_COLAB_FULL_B15_sft_only_seed{seed}",
    "3B": "007_p{passes}_COLAB_FULL_B3_sft_only_seed{seed}",
    "7B": "025_p{passes}_COLAB_FULL_B7_sft_only_seed{seed}",
}
envelope_roster = {}
for scale, pattern in plain_tags.items():
    routes = {}
    for passes in (2, 8, 32):
        tags = (MCSFT_P8_TAGS[scale] if passes == 8 else pattern.replace("{passes}", str(passes)))
        routes[f"Plain MMLU MC-SFT (p{passes})"] = [tags.format(seed=seed) for seed in (0, 1, 2)]
    routes["Warm MMLU MC-SFT (p8)"] = [WARM_MCSFT_TAGS[scale][seed] for seed in (0, 1, 2)]
    if scale == "7B":
        for label, tag in (("Plain", "024_p16"), ("Warm", "024_wsft_p16")):
            routes[f"{label} MMLU MC-SFT (p16)"] = [
                f"{tag}_COLAB_FULL_B7_sft_only_seed{seed}" for seed in (0, 1, 2)
            ]
    envelope_roster[scale] = routes

envelope_scopes = ["overall", *SUBJECT_ORDER]
envelope_score_rows = []
envelope_inputs = set()
for scale, routes in envelope_roster.items():
    grpo_tags = [QWEN_PIPELINES[scale]["tag"].format(seed=seed) for seed in (0, 1, 2)]
    reference_gold = None
    for route, tags in {"Warm GRPO": grpo_tags, **routes}.items():
        filename = "grpo_preds.jsonl" if route == "Warm GRPO" else "sft_preds.jsonl"
        for seed, tag in enumerate(tags):
            path = RESULTS / tag / filename
            scores, gold = endpoint_accuracy(path, comparison_hashes)
            if reference_gold is None:
                reference_gold = gold
            assert gold == reference_gold, f"{scale} {route} seed {seed}: subject answer sequence differs"
            envelope_inputs.add(path.relative_to(CODE_ROOT).as_posix())
            for scope, score in zip(envelope_scopes, scores):
                envelope_score_rows.append({
                    "scale": scale, "route": route, "seed": seed, "scope": scope, "accuracy": score * 100,
                })
assert envelope_inputs == set(comparison_hashes) and len(envelope_inputs) == 66

envelope_scores = pd.DataFrame(envelope_score_rows)
envelope_means = envelope_scores.groupby(["scale", "route", "scope"], sort=False).accuracy.mean()
envelope_comparisons = []
for scale, routes in envelope_roster.items():
    for scope in envelope_scopes:
        values = {route: envelope_means.loc[(scale, route, scope)] for route in routes}
        best = max(values.values())
        winners = [route for route, value in values.items() if np.isclose(value, best, atol=1e-10, rtol=0)]
        grpo = envelope_means.loc[(scale, "Warm GRPO", scope)]
        envelope_comparisons.append({
            "scale": scale, "scope": scope, "warm_grpo": grpo, "best_mc_sft": best,
            "difference": grpo - best, "selected_routes": "; ".join(winners),
        })
envelope_comparison_df = pd.DataFrame(envelope_comparisons)
assert len(envelope_comparison_df) == 36
display(envelope_comparison_df.round(4))

In [ ]:
def finish_gain_panels(axes, label_size):
    for letter, ax in zip("ABCDE", axes):
        ax.text(-.025, 1.05, letter, transform=ax.transAxes, ha="right",
                fontsize=label_size, fontweight="bold")


def style_zero_guides(axes):
    for ax in axes:
        ax.set_axisbelow(True)
        guide, = ax.lines
        assert np.all(np.asarray(guide.get_ydata()) == 0)
        guide.set_zorder(.75)
        guide.set_linewidth(.6)
        guide.set_color("#888888")
        for tick in ax.yaxis.get_major_ticks():
            if abs(tick.get_loc()) < 1e-10:
                tick.gridline.set_visible(False)

In [ ]:
fig = plt.figure(figsize=(8.3, 8.2), layout="constrained")
grid = fig.add_gridspec(3, 2, height_ratios=[.85, 1, 1])
overall_ax = fig.add_subplot(grid[0, :])
overall = envelope_comparison_df[envelope_comparison_df.scope == "overall"]
values = overall.difference.to_numpy()
bars = overall_ax.bar(range(4), values, width=.52,
                      color=[PALETTE["green"] if v > 0 else PALETTE["red"] for v in values])
for bar, value in zip(bars, values):
    overall_ax.annotate(f"{value:+.1f}", (bar.get_x() + bar.get_width() / 2, value),
                        xytext=(0, 4 if value >= 0 else -4), textcoords="offset points",
                        ha="center", va="bottom" if value >= 0 else "top", fontsize=10)
overall_ax.set_xticks(range(4), list(QWEN_PIPELINES))
overall_ax.set_ylim(-15.5, 5)
overall_ax.set_title("Overall")
finish_axes(overall_ax, ylabel="Difference (percentage points)", zero=True)

for column, scale in enumerate(QWEN_PIPELINES):
    ax = fig.add_subplot(grid[1 + column // 2, column % 2])
    frame = envelope_comparison_df[(envelope_comparison_df.scale == scale) &
                                    (envelope_comparison_df.scope != "overall")]
    values = frame.difference.to_numpy()
    ax.bar(range(8), values, width=.64,
           color=[PALETTE["green"] if v > 0 else PALETTE["red"] for v in values])
    for low, high, tier in [(-.5, 2.5, "easy"), (2.5, 5.5, "medium"), (5.5, 7.5, "hard")]:
        ax.axvspan(low, high, color=TIER_BACKGROUNDS[tier], alpha=.40, zorder=0)
    ax.set_xticks(range(8), SUBJECT_LABELS, rotation=42, ha="right", fontsize=9.5)
    ax.set_ylim(-15.5, 15.5)
    ax.set_yticks([-15, -10, -5, 0, 5, 10, 15])
    ax.set_title(scale)
    finish_axes(ax, ylabel="Difference (percentage points)" if column % 2 == 0 else None, zero=True)
finish_gain_panels(fig.axes, 9.75)
# Settle the shared layout with both renderers before fixing the figure geometry.
fig.savefig(BytesIO(), format="png", dpi=300, bbox_inches="tight")
fig.savefig(BytesIO(), format="pdf", bbox_inches="tight")
fig.set_layout_engine("none")
fig.canvas.draw()
style_zero_guides(fig.axes)
save_plot(fig, "JF16_qwen_best_tested_mc_sft")

## Figure S2. Subject gains across five training routes
Each panel shows one Qwen2.5 size. Each bar compares a route with the unmodified pretrained base on the same subject and averages the three seed gains. Both MMLU MC-SFT routes use eight passes.

In [ ]:
recipe_subject_values = {}
for scale, runs in endpoints.items():
    base = runs["Pretrained base"][1:] * 100
    recipe_subject_values[scale] = {
        route: (runs[route][:, 1:] * 100 - base).mean(axis=0) for route in route_order
    }

fig, axes = plt.subplots(4, 1, figsize=(8.8, 9.4), sharex=True, sharey=True)
subject_x = np.arange(len(SUBJECT_ORDER))
width = .15
for ax, scale in zip(axes.flat, QWEN_PIPELINES):
    for index, route in enumerate(route_order):
        ax.bar(subject_x + (index - 2) * width, recipe_subject_values[scale][route],
               width, color=ROUTE_COLOURS[route], label=route + " (p8)" if "MC-SFT" in route else route)
    ax.axhline(0, color="#333333", linewidth=1)
    ax.axvspan(-.5, 2.5, color=TIER_BACKGROUNDS["easy"], alpha=.45, zorder=0)
    ax.axvspan(2.5, 5.5, color=TIER_BACKGROUNDS["medium"], alpha=.40, zorder=0)
    ax.axvspan(5.5, 7.5, color=TIER_BACKGROUNDS["hard"], alpha=.45, zorder=0)
    ax.set_xticks(subject_x, SUBJECT_LABELS, rotation=24, ha="right", fontsize=10.5)
    ax.set_title(scale)
    ax.grid(axis="y", alpha=.18)
fig.supylabel("Gain over pretrained base (percentage points)", x=.012, fontsize=11)
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", ncol=2, bbox_to_anchor=(.53, .995), fontsize=10)
fig.subplots_adjust(left=.095, right=.99, top=.895, bottom=.10, hspace=.36)
finish_gain_panels(axes, 10.2)
fig.canvas.draw()
style_zero_guides(axes)
save_plot(fig, "JF20_qwen_subject_recipe_gains_v40")

## Additional figures from the paper and appendix
The remaining panels use the same frozen runs as the paper. The matched full-parameter comparison includes the recorded early stop.

In [ ]:
import textwrap
from matplotlib.container import BarContainer, ErrorbarContainer
from matplotlib.collections import PathCollection, FillBetweenPolyCollection

TIER_COLOURS = {"easy": PALETTE["green"], "medium": PALETTE["blue"],
                "hard": PALETTE["red"], "overall": PALETTE["grey"]}

with (CODE_ROOT / "analysis/figure_sources/054/additional_inputs.json").open() as handle:
    additional_inputs = json.load(handle)
for name, expected in additional_inputs.items():
    path = CODE_ROOT / name
    assert hashlib.sha256(path.read_bytes()).hexdigest() == expected, f"Input differs: {path}"

def read_json(path):
    path = Path(path)
    with path.open() as handle:
        return json.load(handle)


def read_jsonl(path, expected_rows=None):
    path = Path(path)
    with path.open() as handle:
        rows = [json.loads(line) for line in handle if line.strip()]
    if expected_rows is not None:
        assert len(rows) == expected_rows, f"{path}: expected {expected_rows}, found {len(rows)}"
    return rows



def mean_range(values):
    values = np.asarray(values, dtype=float)
    return float(values.mean()), float(values.min()), float(values.max())


def subject_accuracy(rows, subject):
    subset = [row for row in rows if row.get("subject") == subject]
    assert subset, f"no rows found for subject {subject}"
    return accuracy(subset)

def tier_accuracy(rows, tier):
    subset = [row for row in rows if row.get("tier") == tier]
    assert subset, f"no rows found for tier {tier}"
    return accuracy(subset)

In [ ]:
FIGURE_SIZES = {
    'JF04': (10.8, 5.6),
    'JF06': (10.4, 5.5),
    'JF07': (10.4, 8.3),
    'JF08': (8.8, 5.2),
    'JF09': (9.5, 5.3),
    'JF10': (10.8, 5.7),
    'JF13': (10.2, 4.9),
    'JF14': (10.8, 6.0),
    'JF15': (10.8, 7.2),
    'JF21': (10.4, 6.0),
    'JF22': (10.8, 6.8),
    'JF23': (10.4, 5.7),
    'JF24': (10.8, 5.8),
}

def legend_style(legend, fontsize=12):
    if legend is not None:
        for text in legend.get_texts():
            text.set_fontsize(fontsize)


def style_figure(fig, key):
    if key not in FIGURE_SIZES:
        return
    fig.set_size_inches(*FIGURE_SIZES[key])
    for text in fig.findobj(Text):
        if text.get_text():
            text.set_fontsize(max(text.get_fontsize(), 12))
    for ax in fig.axes:
        ax.tick_params(labelsize=12)
        ax.xaxis.label.set_fontsize(13)
        ax.yaxis.label.set_fontsize(13)
        if len(ax.get_ylabel()) > 46 and '\n' not in ax.get_ylabel():
            ax.set_ylabel('\n'.join(textwrap.wrap(ax.get_ylabel(), 43)), fontsize=13)
        if len(ax.get_xlabel()) > 50 and '\n' not in ax.get_xlabel():
            ax.set_xlabel('\n'.join(textwrap.wrap(ax.get_xlabel(), 43)), fontsize=13)
        title = ax.get_title()
        if len(title) > 37 and '\n' not in title:
            ax.set_title('\n'.join(textwrap.wrap(title, 36)), fontsize=14, pad=12)
        else:
            ax.title.set_fontsize(14)
        legend_style(ax.get_legend())
    for legend in fig.legends:
        legend_style(legend)
    if key == 'JF15':
        fig.subplots_adjust(left=0.14, right=0.95, top=0.9, bottom=0.10, hspace=0.65, wspace=0.42)
    elif key == 'JF07':
        fig.subplots_adjust(left=0.085, right=0.985, top=0.94, bottom=0.09, hspace=0.55, wspace=0.33)
        ax = fig.axes[0]
        handles, labels = ax.get_legend_handles_labels()
        ax.legend(handles, labels, loc='lower right', fontsize=10.5, ncol=1, labelspacing=0.2, borderaxespad=0.25)
    elif key == 'JF21':
        fig.tight_layout(rect=(0, 0, 1, 0.89), w_pad=2.5)
    elif key == 'JF22':
        fig.set_layout_engine('constrained', w_pad=0.09, h_pad=0.09)
    else:
        fig.tight_layout(pad=1.0, w_pad=2.4, h_pad=2)
    if key == 'JF04':
        for text in list(fig.axes[1].texts):
            if text.get_text() in ['Warm GRPO', 'Cold GRPO']:
                text.remove()
        fig.axes[1].legend(fig.axes[1].lines[:2], ['Warm GRPO', 'Cold GRPO'], loc='lower right', fontsize=12)
    if key == 'JF06':
        for text in fig.axes[1].texts:
            if 'A1 stop at 50' in text.get_text():
                text.set_text('A1 stop\nat 50')
    if key == 'JF23':
        ax = fig.axes[0]
        handles, labels = ax.get_legend_handles_labels()
        ax.get_legend().remove()
        fig.legend(handles, labels, loc='upper center', bbox_to_anchor=(0.53, 0.995), ncol=2, fontsize=12)
        fig.tight_layout(rect=(0, 0, 1, 0.87), w_pad=2.4)
    if key == 'JF24':
        fig.axes[1].set_axisbelow(True)
        fig.axes[1].set_title('Matched recipe and\nseed-matched start', fontsize=14)
        fig.tight_layout(w_pad=2.4)
    if key == 'JF14':
        for text in fig.axes[0].texts:
            if isinstance(text, mpl.text.Annotation):
                text.set_ha('right')
                offset = -25 if 'Seed 1' in text.get_text() else -10 if 'Seed 0' in text.get_text() else 9
                text.set_position((2 if 'Seed 1' in text.get_text() else -8, offset))
                text.set_va('top' if offset < 0 else 'bottom')
        fig.axes[1].set_ylim(top=24)
        for text in fig.axes[1].texts:
            if text.get_text().startswith('Training rewards'):
                text.set_position((0.23, 0.96))
                text.set_ha('center')
            if text.get_text().startswith('Answer-letter probes'):
                text.set_text('Answer-letter probes\n(percentage points)')
                text.set_position((0.745, 0.96))
                text.set_ha('center')
        fig.axes[0].set_xlabel('Selected minus final-window\ntraining reward (×100)', fontsize=13)
        fig.axes[1].set_xticks(range(4), ['Accuracy\nreward', 'Total\nreward', 'Overall\nprobe', 'Hard\nprobe'], rotation=0)
        fig.tight_layout(w_pad=3)
    if key == 'JF21':
        for ax in fig.axes:
            for text in ax.texts:
                if text.get_text() in ['+1.1', '+1.6']:
                    text.set_fontsize(10)
    clarify_labels(fig, key)
    refine_layout(fig, key)
    fig.canvas.draw()


def clarify_labels(fig, key):
    if key == 'JF10':
        for ax in fig.axes:
            for text in ax.texts:
                if 'Registered ±2-point band' in text.get_text():
                    text.set_text('±2-point reference band; not a CI')
                    text.set_fontsize(11)
        ax = fig.axes[0]
        legend = ax.get_legend()
        for text in legend.get_texts():
            if text.get_text() == 'Warm GRPO':
                text.set_text('Warm: relative to GSM8K-SFT')
            if text.get_text() == 'Cold GRPO':
                text.set_text('Cold: relative to pretrained base')
        legend_style(legend, 10.5)
    if key == 'JF07':
        for ax in fig.axes[:2]:
            for text in ax.texts:
                if text.get_text() == 'Chance':
                    text.set_text('25% uniform choice')
                    text.set_fontsize(11)
    if key == 'JF23':
        for ax in fig.axes:
            for text in ax.texts:
                if text.get_text().endswith('% format'):
                    text.set_text(text.get_text().replace(' format', ''))
        fig.text(0.5, 0.936, 'Labels: format share of total reward', ha='center', va='top', fontsize=12)
        fig.tight_layout(rect=(0, 0, 1, 0.85), w_pad=2.4)
    if key == 'JF24':
        fig.text(0.5, 0.005, 'Zero-KL first updates are included in means, but cannot appear on a log axis.', ha='center', va='bottom', fontsize=11)
        fig.tight_layout(rect=(0, 0.045, 1, 1), w_pad=2.4)
    fig.canvas.draw()


def refine_layout(fig, key):
    if key == "JF09":
        fig.axes[0].legend(handles=SEED_HANDLES, ncol=1, loc="lower right", fontsize=12)
    if key == "JF21":
        fig.canvas.draw()
        renderer = fig.canvas.get_renderer()
        totals = []
        for ax in fig.axes:
            bars = [p for p in ax.patches if isinstance(p, mpl.patches.Rectangle)]
            bar_width = min(p.get_window_extent(renderer).width for p in bars)
            for text in ax.texts:
                if text.get_text().startswith("total "):
                    text.set_fontsize(10.5)
                    totals.append((text, bar_width - 2))
        size = min(10.5 * limit / text.get_window_extent(renderer).width
                   for text, limit in totals)
        for text, _ in totals:
            text.set_fontsize(min(10.5, size))
    if key == "JF15":
        # reserve label space without extrapolating reference line above 100%
        fig.axes[1].set_xlim(48, 106)
        fig.axes[1].set_ylim(0, 88)
        fig.axes[1].set_yticks([0, 20, 40, 60, 80])
    if key == "JF23":
        # close the header gap without resizing the plotting areas
        shift = .35 / fig.get_figheight()
        fig.legends[0].set_bbox_to_anchor((.53, .995 - shift))
        for text in fig.texts:
            if text.get_text() == "Labels: format share of total reward":
                text.set_y(.936 - shift)
    if key == "JF24":
        for text in fig.axes[0].texts:
            if text.get_text().startswith("3B full model stopped"):
                text.set_position((150, 2.0))
    if key == "JF14":
        for text in fig.axes[1].texts:
            if text.get_text().startswith("Answer-letter probes"):
                text.set_fontsize(11)
    fig.canvas.draw()

In [ ]:
def finish_paper_plot(fig, figure_id, stem):
    if figure_id == "JF03":
        style_exposure(fig)
    else:
        style_figure(fig, figure_id)
        panels = {"JF13": 2, "JF15": 3, "JF21": 2, "JF22": 2, "JF23": 2, "JF24": 2}
        for letter, ax in zip("ABC", fig.axes[:panels.get(figure_id, 0)]):
            ax.text(-.025, 1.05, letter, transform=ax.transAxes, ha="right",
                    fontsize=12, fontweight="bold")

    # Let constrained layouts settle with both renderers before freezing axes.
    if fig.get_layout_engine() is not None:
        fig.savefig(BytesIO(), format="png", dpi=300, bbox_inches="tight")
        fig.savefig(BytesIO(), format="pdf", bbox_inches="tight")
        fig.set_layout_engine("none")
    for ax in fig.axes:
        has_bars = any(isinstance(c, BarContainer) for c in ax.containers)
        if has_bars:
            ax.set_axisbelow(True)
        for line in ax.lines:
            coverage = figure_id == "JF15" and line.get_linestyle() == ":" and len(line.get_xdata()) == 101
            if (line.get_transform() is not ax.transData or coverage) and min(mpl.colors.to_rgba(line.get_color())[:3]) < .95:
                line.set_zorder(.75 if has_bars else 1.75)
            elif figure_id == "JF05" and line.get_linewidth() >= 1.8 and (line.get_alpha() is None or line.get_alpha() >= .5):
                line.set_zorder(2.5)

    if figure_id in {"JF03", "JF04"}:
        ax = fig.axes[1 if figure_id == "JF03" else 0]
        for points in ax.collections:
            if isinstance(points, PathCollection):
                points.set_zorder(3)
        for bars in ax.containers:
            if isinstance(bars, ErrorbarContainer):
                mean, caps, stems = bars.lines
                mean.set_zorder(4.1)
                for cap in caps:
                    cap.set_zorder(3.5)
                    cap.set_color("#888888")
                    cap.set_markeredgecolor("#888888")
                    cap.set_markeredgewidth(.8)
                for line in stems:
                    line.set_zorder(3.5)
                    line.set_color("#888888")
                    line.set_linewidth(.8)

    for i in {"JF21": (0, 1), "JF14": (1,), "JF07": (2,)}.get(figure_id, ()):
        fig.canvas.draw()
        ax = fig.axes[i]
        zero, = [line for line in ax.lines if line.get_transform() is not ax.transData
                 and np.all(np.asarray(line.get_ydata()) == 0)]
        zero.set_color("#888888")
        zero.set_linewidth(.6)
        for tick in ax.yaxis.get_major_ticks():
            if abs(tick.get_loc()) < 1e-10:
                tick.gridline.set_visible(False)
    for i in {"JF10": (1,), "JF09": (0,), "JF04": (1,), "JF03": (0,), "JF05": (0,)}.get(figure_id, ()):
        ax = fig.axes[i]
        for band in ax.collections:
            if isinstance(band, FillBetweenPolyCollection):
                band.set_zorder(1.6)
        if figure_id in {"JF10", "JF09", "JF05"}:
            guide, = [line for line in ax.lines if line.get_transform() is not ax.transData]
            guide.set_zorder(1.55)
    if figure_id == "JF05":
        ax = fig.axes[0]
        fig.canvas.draw()
        legend_box = ax.get_legend().get_window_extent(fig.canvas.get_renderer())
        guide, = [line for line in ax.lines if line.get_label() == "Original GRPO budget"]
        top = ax.transAxes.inverted().transform((legend_box.x0, legend_box.y0 - 3 * fig.dpi / 72))[1]
        guide.set_ydata([0, top])
    save_plot(fig, stem)
    plt.close(fig)

In [ ]:
def style_exposure(fig):
    fig.set_size_inches(10.4, 5.1)
    for text in fig.findobj(Text):
        if text.get_text():
            text.set_fontsize(max(text.get_fontsize(), 12))
    for ax in fig.axes:
        ax.tick_params(labelsize=12)
        ax.xaxis.label.set_fontsize(13)
        ax.yaxis.label.set_fontsize(13)
        if len(ax.get_ylabel()) > 46 and '\n' not in ax.get_ylabel():
            ax.set_ylabel('\n'.join(textwrap.wrap(ax.get_ylabel(), 43)), fontsize=13)
        if len(ax.get_xlabel()) > 50 and '\n' not in ax.get_xlabel():
            ax.set_xlabel('\n'.join(textwrap.wrap(ax.get_xlabel(), 43)), fontsize=13)
        title = ax.get_title()
        if len(title) > 37 and '\n' not in title:
            ax.set_title('\n'.join(textwrap.wrap(title, 36)), fontsize=14, pad=12)
        else:
            ax.title.set_fontsize(14)
        if ax.get_legend() is not None:
            for text in ax.get_legend().get_texts():
                text.set_fontsize(12)
    fig.tight_layout(pad=1.0, w_pad=2.4, h_pad=2)
    fig.axes[1].set_title('7B exposure endpoints\n(zoomed y-axis)', fontsize=14)
    fig.canvas.draw()
    fig.axes[1].legend(handles=SEED_HANDLES, ncol=1, loc='lower right', fontsize=11,
                           labelspacing=.3, borderaxespad=.4)
    for letter, ax in zip('AB', fig.axes):
        ax.text(-.025, 1.05, letter, transform=ax.transAxes,
                ha='right', fontsize=12, fontweight='bold')
    fig.canvas.draw()

    # Connect the four observed means at the tested pass counts.
    ax = fig.axes[1]
    means = [bars.lines[0] for bars in ax.containers if isinstance(bars, ErrorbarContainer)]
    assert len(means) == 4
    ax.set_autoscale_on(False)
    ax.plot([m.get_xdata()[0] for m in means], [m.get_ydata()[0] for m in means],
            color="#222222", linewidth=1.5, zorder=2.5)

## Figure 18. Accuracy across supervised training passes

In [ ]:
exposure_027_path = RESULTS / "027_comparison_summary.json"
exposure_025_path = RESULTS / "025_comparison_summary.json"
exposure_027 = read_json(exposure_027_path)
exposure_025 = read_json(exposure_025_path)

exposure_rows = []
for endpoint in exposure_027["endpoints"]:
    exposure_rows.append({
        "scale": endpoint["scale"],
        "exposure": endpoint["exposure"],
        "mean": endpoint["mean"],
        "minimum": endpoint["minimum"],
        "maximum": endpoint["maximum"],
    })
exposure_df = pd.DataFrame(exposure_rows)

fig, axes = plt.subplots(1, 2, figsize=(12.0, 4.5))
exposure_order = ["p2", "p8", "p32"]
for scale, colour, marker in (("0.5B", PALETTE["blue"], "o"),
                              ("1.5B", PALETTE["green"], "s"),
                              ("3B", PALETTE["orange"], "D")):
    frame = exposure_df[exposure_df.scale == scale].set_index("exposure").loc[exposure_order]
    axes[0].plot([2, 8, 32], frame["mean"] * 100, marker=marker,
                 color=colour, linewidth=2, label=scale)
    axes[0].fill_between([2, 8, 32], frame["minimum"] * 100, frame["maximum"] * 100,
                         color=colour, alpha=.1)
finish_axes(axes[0], ylabel="MMLU generated accuracy (%)", xlabel="Dataset passes")
axes[0].set_xscale("log", base=2)
axes[0].set_xticks([2, 8, 32], ["p2", "p8", "p32"])
axes[0].set_title("0.5B to 3B exposure endpoints")
axes[0].legend()

curve_7b = exposure_025["exposure_curve"]
for exposure, x in (("p2", 2), ("p8", 8), ("p16", 16), ("p32", 32)):
    values = np.array([row["acc"] for row in curve_7b[exposure]]) * 100
    mean, low, high = mean_range(values)
    axes[1].scatter(np.full(3, x), values,
                    color=[SEED_COLOURS[int(row["seed"])] for row in curve_7b[exposure]], s=34)
    axes[1].errorbar(x, mean, yerr=[[mean - low], [high - mean]],
                     fmt="D", color="#222222", capsize=4)
finish_axes(axes[1], ylabel="MMLU generated accuracy (%)", xlabel="Dataset passes")
axes[1].set_xscale("log", base=2)
axes[1].set_xticks([2, 8, 16, 32], ["p2", "p8", "p16", "p32"])
axes[1].set_title("7B exposure endpoints")
axes[1].legend(handles=SEED_HANDLES, ncol=3, loc="lower right", fontsize=8)
axes[1].set_ylim(axes[1].get_ylim()[0] - 1.8, axes[1].get_ylim()[1] + 1.0)
fig.tight_layout()

finish_paper_plot(fig, "JF03", "JF03_supervised_exposure")

## Figure 12. Effect of supervised mathematics training before GRPO

In [ ]:
warm_summary_path = RESULTS / "s4_comparison_summary.json"
warm_summary = read_json(warm_summary_path)
warm_rows = []
for scale, payload in warm_summary["scales"].items():
    for row in payload["seeds"]:
        warm_rows.append({
            "scale": scale,
            "params": QWEN_PIPELINES[scale]["params"],
            "seed": int(row["seed"]),
            "overall_delta": float(row["warm_minus_base"]),
            "hard_delta": float(row["hard_warm_minus_base"]),
            "reading": row["reading"],
        })
warm_df = pd.DataFrame(warm_rows).sort_values(["params", "seed"])

base_route_tags = {
    "0.5B": "005_COLAB_FULL_B05_base_grpo_seed{seed}",
    "1.5B": "012_COLAB_FULL_B15_base_grpo_seed{seed}",
    "3B": "004_COLAB_FULL_B3_base_grpo_seed{seed}",
    "7B": "013_COLAB_FULL_B7_base_grpo_seed{seed}",
}
format_rows = []

for scale, spec in QWEN_PIPELINES.items():
    for seed in (0, 1, 2):
        warm_path = RESULTS / spec["tag"].format(seed=seed) / "grpo_eval.json"
        base_path = RESULTS / base_route_tags[scale].format(seed=seed) / "grpo_eval.json"

        format_rows.extend([
            {"scale": scale, "params": spec["params"], "seed": seed,
             "route": "Warm GRPO", "format": read_json(warm_path)["format_compliance"]},
            {"scale": scale, "params": spec["params"], "seed": seed,
             "route": "Cold GRPO", "format": read_json(base_path)["format_compliance"]},
        ])
format_df = pd.DataFrame(format_rows)

fig, axes = plt.subplots(1, 2, figsize=(13.2, 5.0),
                         gridspec_kw={"width_ratios": [1.12, 1]})
positions = np.arange(len(QWEN_PIPELINES))
for offset, metric, label, colour, marker in (
    (-.10, "overall_delta", "Overall MMLU", TIER_COLOURS["overall"], "o"),
    (.10, "hard_delta", "Hard tier", TIER_COLOURS["hard"], "D"),
):
    means = []
    for index, scale in enumerate(QWEN_PIPELINES):
        frame = warm_df[warm_df.scale == scale].sort_values("seed")
        values = frame[metric].to_numpy() * 100
        x_value = positions[index] + offset
        axes[0].scatter(np.full(3, x_value), values, s=34, facecolor="white",
                        edgecolor=colour, linewidth=1.2, alpha=.9, zorder=3)
        mean, low, high = mean_range(values)
        means.append(mean)
        axes[0].errorbar(x_value, mean, yerr=[[mean - low], [high - mean]],
                         fmt=marker, color=colour, capsize=4, markersize=6, zorder=4)
    axes[0].plot(positions + offset, means, color=colour, linewidth=1.8, alpha=.85,
                 label=label)
axes[0].set_xticks(positions, list(QWEN_PIPELINES))
finish_axes(axes[0], ylabel="Warm minus cold GRPO (percentage points)",
            xlabel="Model scale", zero=True)
axes[0].set_title("Seed-matched route difference")
axes[0].legend(loc="upper right")

for route, colour, marker in (
    ("Warm GRPO", ROUTE_COLOURS["Warm GRPO"], "o"),
    ("Cold GRPO", ROUTE_COLOURS["Cold GRPO"], "s"),
):
    means, lows, highs = [], [], []
    for scale in QWEN_PIPELINES:
        values = format_df[(format_df.scale == scale) &
                           (format_df.route == route)].format.to_numpy() * 100
        mean, low, high = mean_range(values)
        means.append(mean)
        lows.append(low)
        highs.append(high)
    axes[1].plot(positions, means, color=colour, marker=marker, linewidth=2.2,
                 markersize=6)
    axes[1].fill_between(positions, lows, highs, color=colour, alpha=.13)
    label_y = means[-1] + (1.5 if route == "Warm GRPO" else -1.5)
    axes[1].text(positions[-1] + .08, label_y, route, color=colour,
                 va="center", fontsize=9)
axes[1].set_xlim(-.15, positions[-1] + .78)
axes[1].set_xticks(positions, list(QWEN_PIPELINES))
axes[1].set_ylim(-3, 103)
finish_axes(axes[1], ylabel="Format-compliant outputs (%)", xlabel="Model scale")
axes[1].set_title("Absolute output-format viability")
for label, ax in zip(("A", "B"), axes):
    ax.text(-.10, 1.05, label, transform=ax.transAxes, fontsize=12, fontweight="bold")
fig.tight_layout()

finish_paper_plot(fig, "JF04", "M02_warm_start_route")

## Figure S1. 7B accuracy through training

In [ ]:
GRID = (0, 200, 400, 500, 600, 800, 1000)

def grid_result_dir(seed, step):
    if step == 0:
        return RESULTS / f"013_COLAB_FULL_B7_sft_grpo_seed{seed}"
    return RESULTS / f"022_smooth_B7_seed{seed}_step{step}"


trajectory_rows = []

for seed in (0, 1, 2):
    for step in GRID:
        result_dir = grid_result_dir(seed, step)
        if step == 0:
            generated_path = result_dir / "sft_preds.jsonl"
            probe_path = result_dir / "sft_logprob_preds.jsonl"
        else:
            generated_path = result_dir / "grpo_preds.jsonl"
            probe_path = result_dir / "grpo_logprob_preds.jsonl"
        generated = read_jsonl(generated_path, 1884)
        probe = read_jsonl(probe_path, 1884)

        trajectory_rows.append({
            "seed": seed,
            "step": step,
            "generated": accuracy(generated),
            "probe": accuracy(probe),
        })

trajectory_df = pd.DataFrame(trajectory_rows)
fig, ax = plt.subplots(figsize=(8.6, 4.8))
for metric, label, colour, marker in (
    ("generated", "Generated accuracy", PALETTE["orange"], "o"),
    ("probe", "Letter probe", PALETTE["blue"], "s"),
):
    summary = trajectory_df.groupby("step")[metric].agg(["mean", "min", "max"]).reset_index()
    ax.plot(summary.step, summary["mean"] * 100, marker=marker, color=colour,
            linewidth=2, label=label)
    ax.fill_between(summary.step, summary["min"] * 100, summary["max"] * 100,
                    color=colour, alpha=.12)
    for seed in (0, 1, 2):
        rows = trajectory_df[trajectory_df.seed == seed]
        ax.plot(rows.step, rows[metric] * 100, color=colour, alpha=.18, linewidth=.8)
ax.axvline(500, color=PALETTE["grey"], linestyle=":", linewidth=1.2,
           label="Original GRPO budget")
finish_axes(ax, ylabel="MMLU accuracy (%)", xlabel="GRPO update")
ax.set_xticks(GRID)
ax.legend(ncol=3, fontsize=9)
fig.tight_layout()

finish_paper_plot(fig, "JF05", "JF05_qwen_7b_trajectory")

## Figure 13. Full-parameter updates compared with LoRA

In [ ]:
matched_paths = {
    "1.5B LoRA": RESULTS / "035_COLAB_FULL_B15_matched_lora_grpo_seed0",
    "1.5B full": RESULTS / "035_COLAB_FULL_B15_matched_full_grpo_seed0",
}

matched_endpoint_rows = []
for label, result_dir in matched_paths.items():
    gen_path = result_dir / "grpo_preds.jsonl"
    probe_path = result_dir / "grpo_logprob_preds.jsonl"
    generated = read_jsonl(gen_path, 1884)
    probe = read_jsonl(probe_path, 1884)

    matched_endpoint_rows.append({
        "mode": label.replace("1.5B ", "").title(),
        "generated": accuracy(generated),
        "probe": accuracy(probe),
    })
matched_endpoint_df = pd.DataFrame(matched_endpoint_rows)

status_paths = {
    ("3B", "Full"): RESULTS / "035_COLAB_FULL_B3_matched_full_grpo_seed0/worker_status.json",
    ("3B", "LoRA"): RESULTS / "035_COLAB_FULL_B3_matched_lora_grpo_seed0/worker_status.json",
    ("1.5B", "Full"): RESULTS / "035_COLAB_FULL_B15_matched_full_grpo_seed0/worker_status.json",
    ("1.5B", "LoRA"): RESULTS / "035_COLAB_FULL_B15_matched_lora_grpo_seed0/worker_status.json",
}

outcome_rows = []
for (scale, mode), path in status_paths.items():
    status = read_json(path)
    outcome_rows.append({
        "scale": scale,
        "mode": mode,
        "updates": 50 if scale == "3B" and mode == "Full" else 500,
        "status": "A1 stop" if scale == "3B" and mode == "Full" else "Completed",
        "peak_reserved_gib": status.get("peak_reserved_gib"),
    })
outcome_df = pd.DataFrame(outcome_rows)

fig, axes = plt.subplots(1, 2, figsize=(12.8, 5.1))
endpoint_order = ["Lora", "Full"]
x_positions = np.arange(len(endpoint_order))
width = .34
for offset, metric, label, colour in (
    (-width / 2, "generated", "Generated", PALETTE["orange"]),
    (width / 2, "probe", "Letter probe", PALETTE["blue"]),
):
    values = matched_endpoint_df.set_index("mode").loc[endpoint_order, metric].to_numpy() * 100
    bars = axes[0].bar(x_positions + offset, values, width, color=colour, label=label)
    for bar, value in zip(bars, values):
        axes[0].text(bar.get_x() + bar.get_width() / 2, value + .7, f"{value:.1f}",
                     ha="center", va="bottom", fontsize=9, color=colour)
axes[0].set_xticks(x_positions, ["LoRA", "Full model"])
axes[0].set_ylim(0, 75)
finish_axes(axes[0], ylabel="MMLU accuracy (%)")
axes[0].set_title("Matched 1.5B endpoints")
axes[0].legend(loc="upper right")

scales = ["3B", "1.5B"]
scale_positions = np.arange(len(scales))
for offset, mode, colour in ((-width / 2, "Full", PALETTE["red"]),
                             (width / 2, "LoRA", PALETTE["blue"])):
    values = [outcome_df.query("scale == @scale and mode == @mode").updates.iloc[0]
              for scale in scales]
    bars = axes[1].bar(scale_positions + offset, values, width, color=colour, label=mode)
    for scale, bar, value in zip(scales, bars, values):
        label = "A1 stop at 50" if scale == "3B" and mode == "Full" else "500"
        axes[1].text(bar.get_x() + bar.get_width() / 2, value + 10, label,
                     ha="center", va="bottom", fontsize=9, color=colour,
                     fontweight="bold" if value < 500 else "normal")
axes[1].axhline(500, color="#777777", linestyle=":", linewidth=1.1)
axes[1].set_xticks(scale_positions, scales)
axes[1].set_ylim(0, 570)
finish_axes(axes[1], ylabel="GRPO updates reached")
axes[1].set_title("Training viability under the same recipe")
axes[1].legend(loc="upper left")
for label, ax in zip(("A", "B"), axes):
    ax.text(-.10, 1.05, label, transform=ax.transAxes, fontsize=12, fontweight="bold")
fig.tight_layout()

finish_paper_plot(fig, "JF06", "JF06_full_model_lora")

## Figure 14. GPT-2 outcomes by model size

In [ ]:
gpt2_summary_path = RESULTS / "048_family_summary.json"
gpt2_summary = read_json(gpt2_summary_path)
assert gpt2_summary["completed"] is True
gpt2_df = pd.DataFrame(gpt2_summary["gpt2_scales"]).sort_values("params")

fig = plt.figure(figsize=(12.6, 7.4))
grid = fig.add_gridspec(2, 2, height_ratios=[1, .82], hspace=.42, wspace=.27)
accuracy_axes = [fig.add_subplot(grid[0, 0]), fig.add_subplot(grid[0, 1])]
change_ax = fig.add_subplot(grid[1, :])
for ax, prefix, title in (
    (accuracy_axes[0], "accuracy", "Generated endpoint accuracy"),
    (accuracy_axes[1], "probe", "Answer-letter probe"),
):
    if prefix == "accuracy":
        columns = (("base_accuracy", "Pretrained base"), ("SFT_accuracy", "GSM8K-SFT"), ("GRPO_accuracy", "Warm GRPO"))
    else:
        columns = (("base_probe", "Pretrained base"), ("SFT_probe", "GSM8K-SFT"), ("GRPO_probe", "Warm GRPO"))
    for (column, label), colour, marker in zip(
        columns, (PALETTE["grey"], PALETTE["blue"], PALETTE["orange"]), ("o", "s", "D")
    ):
        open_marker = prefix == "accuracy" and column == "GRPO_accuracy"
        ax.plot(gpt2_df.params, gpt2_df[column] * 100, marker=marker,
                color=colour, linewidth=2.2, label=label, markersize=6,
                markerfacecolor="white" if open_marker else colour,
                markeredgewidth=1.4)
    ax.axhline(25, color="#999999", linestyle=":", linewidth=1)
    ax.text(gpt2_df.params.iloc[0], 25.5, "Chance", color="#777777", fontsize=8)
    finish_axes(ax, ylabel="MMLU accuracy (%)")
    ax.set_title(title)
    ax.set_ylim(-3 if prefix == "accuracy" else 0, 30)
accuracy_axes[0].legend(ncol=3, loc="lower right", fontsize=8,
                        handlelength=1.5, handletextpad=.4, columnspacing=.9,
                        borderaxespad=.35)
accuracy_axes[0].text(.97, .97, "Open GRPO markers:\nextraction caveat",
                      transform=accuracy_axes[0].transAxes, fontsize=8,
                      color="#666666", ha="right", va="top")

positions = np.arange(len(gpt2_df))
width = .32
for offset, column, label, colour in (
    (-width / 2, "format change", "Format compliance", PALETTE["green"]),
    (width / 2, "coverage change", "Extraction coverage", PALETTE["purple"]),
):
    values = gpt2_df[column].to_numpy() * 100
    bars = change_ax.bar(positions + offset, values, width, color=colour, alpha=.88, label=label)
    for bar, value in zip(bars, values):
        change_ax.text(bar.get_x() + bar.get_width() / 2,
                       value + (.55 if value >= 0 else -.65), f"{value:+.1f}",
                       ha="center", va="bottom" if value >= 0 else "top", fontsize=8)
change_ax.set_xticks(positions, gpt2_df.scale)
finish_axes(change_ax, ylabel="Warm GRPO minus GSM8K-SFT\n(percentage points)",
            xlabel="GPT-2 scale", zero=True)
change_ax.set_ylim(bottom=-6.5)
change_ax.set_title("Format and extraction-coverage changes")
change_ax.legend(ncol=2, loc="upper left")

for ax in accuracy_axes:
    ax.set_xscale("log")
    ax.set_xticks(gpt2_df.params, gpt2_df.scale)
    ax.set_xlabel("GPT-2 scale")
for label, ax in zip(("A", "B", "C"), accuracy_axes + [change_ax]):
    ax.text(-.10, 1.05, label, transform=ax.transAxes, fontsize=12, fontweight="bold")

finish_paper_plot(fig, "JF07", "JF07_gpt2_family")

## Figure 6. Variation across seeds at four Qwen2.5 sizes

In [ ]:
base_grpo_rows = []

for scale, tag in BASE_GRPO_PIPELINES.items():
    for seed in (0, 1, 2):
        path = RESULTS / tag.format(seed=seed) / "grpo_preds.jsonl"
        base_grpo_rows.append({
            "scale": scale,
            "params": QWEN_PIPELINES[scale]["params"],
            "seed": seed,
            "accuracy": accuracy(read_jsonl(path, 1884)),
        })

base_grpo_df = pd.DataFrame(base_grpo_rows)

stability_rows = []
for scale, spec in QWEN_PIPELINES.items():
    route_values = {
        "Warm GRPO": qwen_df.loc[qwen_df.scale == scale, "grpo"].to_numpy(),
        "Cold GRPO": base_grpo_df.loc[base_grpo_df.scale == scale, "accuracy"].to_numpy(),
        "Plain MMLU MC-SFT (p8)": method_df.loc[method_df.scale == scale, "mc_p8"].to_numpy(),
    }
    for route, values in route_values.items():
        stability_rows.append({
            "scale": scale, "params": spec["params"], "route": route,
            "spread": (values.max() - values.min()) * 100,
        })
stability_df = pd.DataFrame(stability_rows)

routes = ["Warm GRPO", "Cold GRPO", "Plain MMLU MC-SFT (p8)"]
colours = [PALETTE["orange"], PALETTE["purple"], PALETTE["green"]]
x = np.arange(len(QWEN_PIPELINES))
width = .24
fig, ax = plt.subplots(figsize=(10.2, 4.8))
for index, (route, colour) in enumerate(zip(routes, colours)):
    frame = stability_df[stability_df.route == route].sort_values("params")
    values = frame.spread.to_numpy()
    positions = x + (index - 1) * width
    bars = ax.bar(positions, values, width, color=colour, alpha=.88, label=route)
    for bar, value in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width() / 2, value + .35, f"{value:.1f}",
                ha="center", va="bottom", fontsize=8)
ax.set_xticks(x, list(QWEN_PIPELINES))
finish_axes(ax, ylabel="Three-seed endpoint range (percentage points)", xlabel="Model scale")
ax.legend(ncol=3, loc="upper right")
ax.set_ylim(0, stability_df.spread.max() + 3)
fig.tight_layout()

finish_paper_plot(fig, "JF08", "JF08_qwen_seed_stability_v40")

## Figure 11. Hard-tier generated accuracy and answer-letter probe accuracy

In [ ]:
hard_margin_df = qwen_df[["scale", "params", "seed"]].copy()
hard_margin_df["margin"] = qwen_df.hard_grpo - qwen_df.hard_grpo_probe

fig, ax = plt.subplots(figsize=(9.8, 4.8))
means, lows, highs, params = [], [], [], []
for scale, spec in QWEN_PIPELINES.items():
    frame = hard_margin_df[hard_margin_df.scale == scale].sort_values("seed")
    values = frame.margin.to_numpy() * 100
    ax.scatter(np.full(3, spec["params"]), values,
               color=[SEED_COLOURS[int(seed)] for seed in frame.seed], s=42, zorder=3)
    mean, low, high = mean_range(values)
    params.append(spec["params"])
    means.append(mean)
    lows.append(low)
    highs.append(high)
    label_positions = {
        "0.5B": (5, -3, "left", "top"),
        "1.5B": (5, -3, "left", "top"),
        "3B": (5, -3, "left", "top"),
        "7B": (-5, 3, "right", "bottom"),
    }
    x_offset, y_offset, horizontal, vertical = label_positions[scale]
    ax.annotate(f"{mean:+.1f}", (spec["params"], mean),
                xytext=(x_offset, y_offset), textcoords="offset points",
                ha=horizontal, va=vertical, fontsize=9)
ax.plot(params, means, color=PALETTE["orange"], marker="D", linewidth=2, zorder=2)
ax.fill_between(params, lows, highs, color=PALETTE["orange"], alpha=.13)
ax.set_xscale("log")
ax.set_xticks(params, list(QWEN_PIPELINES))
finish_axes(ax, ylabel="Generated minus probe accuracy\n(percentage points)",
            xlabel="Model scale", zero=True)
ax.legend(handles=SEED_HANDLES, ncol=3, loc="upper left", fontsize=9)
fig.tight_layout()

finish_paper_plot(fig, "JF09", "JF09_qwen_hard_probe_margin")

## Figure 10. Changes in the answer-letter probe across seeds and checkpoints

In [ ]:
BASE_PROBE_TAGS = {
    "0.5B": "005_COLAB_FULL_B05",
    "1.5B": "012_COLAB_FULL_B15",
    "3B": "004_COLAB_FULL_B3",
    "7B": "013_COLAB_FULL_B7",
}
probe_stability_rows = []

for scale, spec in QWEN_PIPELINES.items():
    for seed in (0, 1, 2):
        warm = qwen_df[(qwen_df.scale == scale) & (qwen_df.seed == seed)].iloc[0]
        probe_stability_rows.append({
            "scale": scale, "params": spec["params"], "seed": seed,
            "route": "Warm GRPO", "probe_delta": warm.probe_delta,
        })
        endpoint_path = (RESULTS / BASE_GRPO_PIPELINES[scale].format(seed=seed) /
                         "grpo_logprob_preds.jsonl")
        base_path = (RESULTS / "base_evals" / BASE_PROBE_TAGS[scale] /
                     "base_logprob_preds.jsonl")
        endpoint = accuracy(read_jsonl(endpoint_path, 1884))
        base = accuracy(read_jsonl(base_path, 1884))

        probe_stability_rows.append({
            "scale": scale, "params": spec["params"], "seed": seed,
            "route": "Cold GRPO", "probe_delta": endpoint - base,
        })
probe_stability_df = pd.DataFrame(probe_stability_rows)

fig, axes = plt.subplots(1, 2, figsize=(12.8, 5.0))
positions = np.arange(len(QWEN_PIPELINES))
for offset, route, colour, marker in (
    (-.12, "Warm GRPO", PALETTE["orange"], "o"),
    (.12, "Cold GRPO", PALETTE["purple"], "s"),
):
    frame = probe_stability_df[probe_stability_df.route == route]
    means = []
    for index, scale in enumerate(QWEN_PIPELINES):
        rows = frame[frame.scale == scale].sort_values("seed")
        values = rows.probe_delta.to_numpy() * 100
        x_value = positions[index] + offset
        axes[0].vlines(x_value, values.min(), values.max(), color=colour,
                       linewidth=2.2, alpha=.55, zorder=2)
        axes[0].scatter(np.full(3, x_value), values, color="white", edgecolor=colour,
                        marker=marker, s=34, linewidth=1.1, zorder=3)
        means.append(values.mean())
        axes[0].scatter(x_value, values.mean(), color=colour, marker=marker,
                        s=48, zorder=4)
    axes[0].plot(positions + offset, means, color=colour, marker=marker,
                 linewidth=1.6, label=route)
axes[0].axhspan(-2, 2, color=PALETTE["light_grey"], alpha=.42, zorder=0)
axes[0].text(.03, .08, "Registered ±2-point band", transform=axes[0].transAxes,
             fontsize=8, color="#666666")
axes[0].set_xticks(positions, list(QWEN_PIPELINES))
finish_axes(axes[0], ylabel="Probe change (percentage points)",
            xlabel="Model scale", zero=True)
axes[0].set_title("Twenty-four fixed core GRPO arms")
axes[0].legend(fontsize=8)

trajectory_probe = trajectory_df.copy()
trajectory_probe["probe_delta"] = trajectory_probe.groupby("seed")["probe"].transform(
    lambda values: values - values.iloc[0]
)
for seed in (0, 1, 2):
    rows = trajectory_probe[trajectory_probe.seed == seed]
    axes[1].plot(rows.step, rows.probe_delta * 100, color=SEED_COLOURS[seed],
                 linewidth=1, alpha=.32)
trajectory_summary = trajectory_probe.groupby("step").probe_delta.agg(["mean", "min", "max"]) * 100
axes[1].fill_between(trajectory_summary.index, trajectory_summary["min"],
                     trajectory_summary["max"], color=PALETTE["blue"], alpha=.15,
                     label="Observed seed range")
axes[1].plot(trajectory_summary.index, trajectory_summary["mean"], color=PALETTE["blue"],
             marker="o", linewidth=2.3, label="Three-seed mean")
axes[1].axhspan(-2, 2, color=PALETTE["light_grey"], alpha=.42, zorder=0)
axes[1].text(.03, .08, "Registered ±2-point band", transform=axes[1].transAxes,
             fontsize=8, color="#666666")
finish_axes(axes[1], ylabel="Probe change from GSM8K-SFT (percentage points)",
            xlabel="GRPO update", zero=True)
axes[1].set_title("7B checkpoint trajectory")
axes[1].legend(loc="upper right", fontsize=8)
for label, ax in zip(("A", "B"), axes):
    ax.text(-.10, 1.05, label, transform=ax.transAxes, fontsize=12, fontweight="bold")
fig.tight_layout()

finish_paper_plot(fig, "JF10", "JF10_qwen_probe_stability_v40")

## Figure S3. Changes in model representations

In [ ]:
representation_path = RESULTS / "040_representation_summary.json"
representation = read_json(representation_path)
assert representation["completed"] is True and representation["model_test_contacts"] == 0
scale_names = {"B05": "0.5B", "B15": "1.5B", "B3": "3B", "B7": "7B"}
representation_rows = []
for key, values in representation["by_scale"].items():
    representation_rows.append({"scale": scale_names[key], "params": QWEN_PIPELINES[scale_names[key]]["params"], **values})
representation_df = pd.DataFrame(representation_rows).sort_values("params")

fig, axes = plt.subplots(1, 2, figsize=(11.8, 4.4))
for ax, mean_col, range_col, title, ylabel, colour in (
    (axes[0], "changed_fraction_mean", "changed_fraction_range",
     "Predictions changed after merging", "Changed development rows (%)", PALETTE["purple"]),
    (axes[1], "accuracy_delta_mean", "accuracy_delta_range",
     "Directional accuracy change", "Merged minus adapter accuracy (percentage points)", PALETTE["green"]),
):
    means = representation_df[mean_col].to_numpy() * 100
    lows = np.array([value[0] for value in representation_df[range_col]]) * 100
    highs = np.array([value[1] for value in representation_df[range_col]]) * 100
    x = np.arange(len(representation_df))
    ax.errorbar(x, means, yerr=np.vstack((means - lows, highs - means)),
                fmt="o", color=colour, capsize=4, markersize=6)
    ax.set_xticks(x, representation_df.scale)
    finish_axes(ax, ylabel=ylabel, zero=mean_col == "accuracy_delta_mean")
    ax.set_title(title)
fig.tight_layout()

finish_paper_plot(fig, "JF13", "JF13_representation_changes")

## Figure 9. Training reward and evaluation outcomes

In [ ]:
SELECTOR_RUNS = {
    0: "017_selector_B05_seed0_step400",
    1: "017_selector_B05_seed1_step450",
    2: "017_selector_B05_seed2_step250",
}
selector_rows = []

for seed, selected_tag in SELECTOR_RUNS.items():
    selected_dir = RESULTS / selected_tag
    final_dir = RESULTS / f"006_COLAB_FULL_B05_sft_grpo_seed{seed}"
    manifest_path = selected_dir / "selector_manifest.json"
    selected_path = selected_dir / "grpo_preds.jsonl"
    final_path = final_dir / "grpo_preds.jsonl"
    manifest = read_json(manifest_path)
    selector_rows.append({
        "seed": seed, "step": int(manifest["selected_step"]),
        "reward_advantage": manifest["exact_score"] - manifest["exact_final_window_score"],
        "held_out_delta": accuracy(read_jsonl(selected_path, 1884)) -
                          accuracy(read_jsonl(final_path, 1884)),
    })

selector_df = pd.DataFrame(selector_rows)

full_trainlog_path = RESULTS / "035_COLAB_FULL_B15_matched_full_grpo_seed0" / "grpo_trainlog.json"
full_start_probe_path = RESULTS / "035b_start_B15_seed0" / "start_logprob_preds.jsonl"
full_end_probe_path = (RESULTS / "035_COLAB_FULL_B15_matched_full_grpo_seed0" /
                       "grpo_logprob_preds.jsonl")
full_steps = [row for row in read_json(full_trainlog_path)
              if row.get("step", 0) <= 500 and "reward" in row]
first_window = [row for row in full_steps if row["step"] <= 50]
last_window = [row for row in full_steps if row["step"] >= 301]
start_probe_rows = read_jsonl(full_start_probe_path, 1884)
end_probe_rows = read_jsonl(full_end_probe_path, 1884)
full_changes = {
    "Accuracy reward": np.mean([row["rewards/accuracy_reward/mean"] for row in last_window]) -
                       np.mean([row["rewards/accuracy_reward/mean"] for row in first_window]),
    "Total reward": np.mean([row["reward"] for row in last_window]) -
                    np.mean([row["reward"] for row in first_window]),
    "Overall probe": accuracy(end_probe_rows) - accuracy(start_probe_rows),
    "Hard probe": tier_accuracy(end_probe_rows, "hard") -
                  tier_accuracy(start_probe_rows, "hard"),
}


fig, axes = plt.subplots(1, 2, figsize=(12.8, 5.1),
                         gridspec_kw={"width_ratios": [1.08, 1]})
selector_x = selector_df.reward_advantage.to_numpy() * 100
selector_y = selector_df.held_out_delta.to_numpy() * 100
x_max = max(selector_x) * 1.18
y_min = min(selector_y) * 1.18
axes[0].fill_between([0, x_max], y_min, 0, color=PALETTE["red"], alpha=.08, zorder=0)
for _, row in selector_df.iterrows():
    axes[0].scatter(row.reward_advantage * 100, row.held_out_delta * 100,
                    color=SEED_COLOURS[int(row.seed)], s=72, edgecolor="white",
                    linewidth=.8, zorder=3)
    axes[0].annotate(f"Seed {int(row.seed)}, step {int(row.step)}",
                     (row.reward_advantage * 100, row.held_out_delta * 100),
                     xytext=(6, 6), textcoords="offset points", fontsize=8)
axes[0].axvline(0, color="#666666", linewidth=1)
axes[0].set_xlim(min(-.5, selector_x.min() - 1), x_max)
axes[0].set_ylim(y_min, max(1.5, selector_y.max() + 1))
axes[0].text(.55, .52, "Higher training reward\nLower held-out accuracy",
             transform=axes[0].transAxes, color=PALETTE["red"], fontsize=9,
             ha="center", fontweight="bold")
finish_axes(axes[0], ylabel="Selected minus final test accuracy\n(percentage points)",
            xlabel="Selected minus final-window training reward (×100)", zero=True)
axes[0].set_title("Reward-selected versus final checkpoints")

labels = list(full_changes)
values = np.array(list(full_changes.values())) * 100
colours = [PALETTE["green"], PALETTE["green"], PALETTE["red"], PALETTE["red"]]
x_positions = np.arange(len(labels))
axes[1].axvspan(-.5, 1.5, color=PALETTE["green"], alpha=.06, zorder=0)
axes[1].axvspan(1.5, 3.5, color=PALETTE["red"], alpha=.06, zorder=0)
bars = axes[1].bar(x_positions, values, color=colours, width=.66, alpha=.9)
for bar, value, colour in zip(bars, values, colours):
    axes[1].text(bar.get_x() + bar.get_width() / 2,
                 value + (.6 if value >= 0 else -.8), f"{value:+.1f}",
                 ha="center", va="bottom" if value >= 0 else "top",
                 fontsize=9, fontweight="bold", color=colour)
axes[1].set_xticks(x_positions, labels, rotation=12)
axes[1].text(.03, .96, "Training rewards\n(change ×100)", transform=axes[1].transAxes,
             color=PALETTE["green"], va="top", fontsize=8)
axes[1].text(.58, .96, "Answer-letter probes\n(percentage points)", transform=axes[1].transAxes,
             color=PALETTE["red"], va="top", fontsize=8)
axes[1].set_ylim(min(values.min() - 3.0, -2.0), max(values.max() + 2.0, 2.0))
finish_axes(axes[1], ylabel="Change over the 500-update run\n(reward ×100; probe percentage points)", zero=True)
axes[1].set_title("Matched 1.5B full-model GRPO")
for label, ax in zip(("A", "B"), axes):
    ax.text(-.10, 1.05, label, transform=ax.transAxes, fontsize=12, fontweight="bold")
fig.tight_layout()

finish_paper_plot(fig, "JF14", "JF14_reward_outcome_disagreement")

## Figure S5. Response length and answer extraction coverage

In [ ]:
INSTRUMENT_PACKAGES = [
    ("Qwen2.5", "0.5B", "006_COLAB_FULL_B05_sft_grpo_seed{seed}"),
    ("Qwen2.5", "1.5B", "009_COLAB_FULL_B15_sft_grpo_seed{seed}"),
    ("Qwen2.5", "3B", "003_COLAB_FULL_B3_sft_grpo_seed{seed}"),
    ("Qwen2.5", "7B", "013_COLAB_FULL_B7_sft_grpo_seed{seed}"),
    ("GPT-2", "124M", "048_COLAB_FULL_GPT2_M124_sft_grpo_seed{seed}"),
    ("GPT-2", "355M", "048_COLAB_FULL_GPT2_M355_sft_grpo_seed{seed}"),
    ("GPT-2", "774M", "048_COLAB_FULL_GPT2_M774_sft_grpo_seed{seed}"),
    ("GPT-2", "XL 1.5B", "046_COLAB_FULL_GPT2_B15_sft_grpo_seed{seed}"),
]
COMPLETION_CAP = 512

instrument_rows = []
instrument_seed_rows = []

for family, label, tag in INSTRUMENT_PACKAGES:
    package_seeds = []
    for seed in (0, 1, 2):
        seed_record = {"family": family, "label": label, "seed": seed}
        for stage in ("sft", "grpo"):
            result_dir = RESULTS / tag.format(seed=seed)
            eval_path = result_dir / f"{stage}_eval.json"
            pred_path = result_dir / f"{stage}_preds.jsonl"
            evaluation = read_json(eval_path)
            predictions = read_jsonl(pred_path, 1884)
            seed_record[f"{stage}_len"] = float(evaluation["mean_len"])
            seed_record[f"{stage}_coverage"] = float(
                1 - evaluation["extraction_failure_rate"]) * 100
            seed_record[f"{stage}_accuracy"] = accuracy(predictions) * 100

        seed_record["guard_fail"] = bool(
            family == "GPT-2" and (
                seed_record["sft_coverage"] < 50
                or seed_record["grpo_coverage"] < 50
                or abs(seed_record["grpo_coverage"] - seed_record["sft_coverage"]) > 2
            )
        )
        package_seeds.append(seed_record)
        instrument_seed_rows.append(seed_record)
    record = {"family": family, "label": label}
    for stage in ("sft", "grpo"):
        for field in ("len", "coverage", "accuracy"):
            record[f"{stage}_{field}"] = float(np.mean(
                [item[f"{stage}_{field}"] for item in package_seeds]))
    record["guard_failures"] = sum(item["guard_fail"] for item in package_seeds)
    instrument_rows.append(record)
instrument_df = pd.DataFrame(instrument_rows)
instrument_seed_df = pd.DataFrame(instrument_seed_rows)

from matplotlib.lines import Line2D

fig = plt.figure(figsize=(10.8, 7.2))
grid = fig.add_gridspec(2, 2, width_ratios=[1.05, 1.2],
                        height_ratios=[1, 1], hspace=.52, wspace=.36)
length_ax = fig.add_subplot(grid[:2, 0])
qwen_ax = fig.add_subplot(grid[0, 1])
gpt_ax = fig.add_subplot(grid[1, 1])
positions = np.arange(len(instrument_df))[::-1]
height = .36
for offset, stage, colour in ((height / 2, "sft", PALETTE["blue"]),
                               (-height / 2, "grpo", PALETTE["orange"])):
    length_ax.barh(positions + offset, instrument_df[f"{stage}_len"],
                   height=height, color=colour, alpha=.9)
length_ax.axvline(COMPLETION_CAP, color=PALETTE["red"], linestyle="--", linewidth=1.1)
length_ax.text(COMPLETION_CAP - 8, 7.75, "512-token cap",
               color=PALETTE["red"], fontsize=9, ha="right", va="top")
length_ax.axhline(3.5, color="#999999", linewidth=.9)
length_ax.set_yticks(positions, [f"{'Qwen' if family == 'Qwen2.5' else 'GPT-2'} {label}"
                                for family, label in zip(instrument_df.family, instrument_df.label)])
length_ax.set_xlim(0, 550)
length_ax.set_xlabel("Mean response length (tokens)")
length_ax.set_title("Response length", loc="center")
length_ax.grid(axis="x", color="#E8E8E8", linewidth=.7)

gpt_label_positions = {"124M": (70, 13), "355M": (43, 9.5),
                       "774M": (76, 22), "XL 1.5B": (53, 21)}
for family, ax in (("Qwen2.5", qwen_ax), ("GPT-2", gpt_ax)):
    seeds = instrument_seed_df[instrument_seed_df.family == family]
    means = instrument_df[instrument_df.family == family]
    for stage, colour, filled in (("sft", PALETTE["blue"], False),
                                   ("grpo", PALETTE["orange"], True)):
        ax.scatter(seeds[f"{stage}_coverage"], seeds[f"{stage}_accuracy"],
                    s=15, facecolor=colour if filled else "white", edgecolor=colour,
                    linewidth=.65, alpha=.3, zorder=2)
    for _, row in means.iterrows():
        ax.annotate("", xy=(row.grpo_coverage, row.grpo_accuracy),
                     xytext=(row.sft_coverage, row.sft_accuracy),
                     arrowprops=dict(arrowstyle="->", color="#646464", linewidth=1.1),
                     zorder=3)
        ax.scatter(row.sft_coverage, row.sft_accuracy, s=37, facecolor="white",
                    edgecolor=PALETTE["blue"], linewidth=1.3, zorder=4)
        ax.scatter(row.grpo_coverage, row.grpo_accuracy, s=53,
                    color=PALETTE["orange"], zorder=5)
        if family == "Qwen2.5":
            ax.annotate(row.label, (row.grpo_coverage, row.grpo_accuracy),
                         xytext=(7, 0), textcoords="offset points", fontsize=9,
                         va="center")
        else:
            ax.annotate(row.label, (row.grpo_coverage, row.grpo_accuracy),
                         xytext=gpt_label_positions[row.label], fontsize=9,
                         arrowprops=dict(arrowstyle="-", color="#999999", linewidth=.7))
    coverage_axis = np.linspace(0, 100, 101)
    ax.plot(coverage_axis, coverage_axis * .25, color=PALETTE["grey"],
             linestyle=":", linewidth=.9, zorder=1)
    ax.set_ylabel("Generated accuracy (%)")
    ax.grid(color="#E8E8E8", linewidth=.7)

qwen_ax.set_title("Qwen2.5", loc="center")
qwen_ax.set_xlim(48, 110)
qwen_ax.set_ylim(12, 88)
qwen_ax.set_xticks([50, 70, 90, 100])
qwen_ax.set_yticks([20, 40, 60, 80])
gpt_ax.set_title("GPT-2 (zoomed axes)", loc="center")
gpt_ax.set_xlim(40, 84)
gpt_ax.set_ylim(8.5, 24)
gpt_ax.set_xticks([40, 50, 60, 70, 80])
gpt_ax.set_yticks([10, 15, 20])
gpt_ax.set_xlabel("Extraction coverage (%)")
gpt_ax.axvline(50, color=PALETTE["red"], linestyle="--", linewidth=.9)
gpt_ax.text(50.8, 23.3, "50% floor", color=PALETTE["red"], fontsize=8.5, va="top")

stage_handles = [
    Line2D([], [], marker="o", linestyle="none", markerfacecolor="white",
           markeredgecolor=PALETTE["blue"], markeredgewidth=1.3, markersize=6,
           label="GSM8K-SFT"),
    Line2D([], [], marker="o", linestyle="none", color=PALETTE["orange"],
           markersize=6, label="Warm GRPO"),
]
fig.legend(handles=stage_handles, loc="upper center", ncol=2,
            bbox_to_anchor=(.55, .995), fontsize=10)
fig.subplots_adjust(left=.125, right=.975, top=.91, bottom=.035)

finish_paper_plot(fig, "JF15", "JF15_length_and_coverage")

## Figure 4. Contributions of each difficulty tier to the overall gain

In [ ]:
tier_names = ["easy", "medium", "hard"]
tier_colours = {tier: TIER_COLOURS[tier] for tier in tier_names}
tier_comparisons = {
    "Warm GRPO minus GSM8K-SFT": {},
    "Warm GRPO minus Plain MMLU MC-SFT (p8)": {},
}

# weight each tier by its share of the full 1,884-question evaluation
for scale in QWEN_PIPELINES:
    seed_contributions = {name: [] for name in tier_comparisons}
    for seed in (0, 1, 2):
        warm_dir = RESULTS / QWEN_PIPELINES[scale]["tag"].format(seed=seed)
        warm_path = warm_dir / "grpo_preds.jsonl"
        sft_path = warm_dir / "sft_preds.jsonl"
        mc_path = RESULTS / MCSFT_P8_TAGS[scale].format(seed=seed) / "sft_preds.jsonl"
        rows_by_name = {
            "warm": read_jsonl(warm_path, 1884),
            "sft": read_jsonl(sft_path, 1884),
            "mc": read_jsonl(mc_path, 1884),
        }

        coverage = {subject: sum(row["subject"] == subject
                                 for row in rows_by_name["warm"])
                    for subject in SUBJECT_ORDER}
        for comparison_rows in (rows_by_name["sft"], rows_by_name["mc"]):
            assert coverage == {subject: sum(row["subject"] == subject
                                             for row in comparison_rows)
                                for subject in SUBJECT_ORDER}
        for name, right_key in (
            ("Warm GRPO minus GSM8K-SFT", "sft"),
            ("Warm GRPO minus Plain MMLU MC-SFT (p8)", "mc"),
        ):
            values = []
            for tier in tier_names:
                subjects = set(SUBJECT_TIERS[tier])
                left_correct = sum(bool(row["correct"]) for row in rows_by_name["warm"]
                                   if row["subject"] in subjects)
                right_correct = sum(bool(row["correct"]) for row in rows_by_name[right_key]
                                    if row["subject"] in subjects)
                values.append((left_correct - right_correct) / 1884 * 100)
            seed_contributions[name].append(values)
    for name, values in seed_contributions.items():
        values = np.asarray(values, dtype=float)
        tier_comparisons[name][scale] = values.mean(axis=0)

fig, axes = plt.subplots(1, 2, figsize=(13.4, 5.4), sharey=True)
x = np.arange(len(QWEN_PIPELINES))
for ax, name in zip(axes, tier_comparisons):
    positive_bottom = np.zeros(len(QWEN_PIPELINES))
    negative_bottom = np.zeros(len(QWEN_PIPELINES))
    for tier_index, tier in enumerate(tier_names):
        values = np.array([tier_comparisons[name][scale][tier_index]
                           for scale in QWEN_PIPELINES])
        bottoms = np.where(values >= 0, positive_bottom, negative_bottom)
        bars = ax.bar(x, values, width=.72, bottom=bottoms,
                      color=tier_colours[tier], label=tier.capitalize())
        for bar, value, bottom in zip(bars, values, bottoms):
            if abs(value) >= 1.0:
                ax.text(bar.get_x() + bar.get_width() / 2, bottom + value / 2,
                        f"{value:+.1f}", ha="center", va="center", fontsize=8)
        positive_bottom += np.where(values > 0, values, 0)
        negative_bottom += np.where(values < 0, values, 0)
    for index, scale in enumerate(QWEN_PIPELINES):
        total = tier_comparisons[name][scale].sum()
        if total >= 0:
            label_y = positive_bottom[index] + .6
            label_va = "bottom"
        else:
            label_y = negative_bottom[index] - .6
            label_va = "top"
        ax.text(index, label_y, f"total {total:+.1f}", ha="center",
                va=label_va, fontsize=8, fontweight="bold")
    ax.set_xticks(x, list(QWEN_PIPELINES))
    finish_axes(ax, xlabel="Model scale", zero=True)
    ax.set_title(name)
axes[0].set_ylabel("Contribution to overall accuracy change (percentage points)")
axes[0].set_ylim(-13.5, 23)
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, ncol=3, loc="upper center", bbox_to_anchor=(.5, .98))
fig.tight_layout(rect=(0, 0, 1, .91))

finish_paper_plot(fig, "JF21", "JF21_qwen_tier_gain_contributions_v40")

## Figure S6. Answer-letter probe scores by subject and model family

In [ ]:
GPT2_PROBE_PIPELINES = {
    "124M": {"params": .124, "tag": "048_COLAB_FULL_GPT2_M124_sft_grpo_seed{seed}"},
    "355M": {"params": .355, "tag": "048_COLAB_FULL_GPT2_M355_sft_grpo_seed{seed}"},
    "774M": {"params": .774, "tag": "048_COLAB_FULL_GPT2_M774_sft_grpo_seed{seed}"},
    "XL 1.5B": {"params": 1.5, "tag": "046_COLAB_FULL_GPT2_B15_sft_grpo_seed{seed}"},
}


def endpoint_subject_probe(pipelines):
    matrix = np.zeros((len(SUBJECT_ORDER), len(pipelines)))
    for column, (scale, spec) in enumerate(pipelines.items()):
        seed_values = []
        for seed in (0, 1, 2):
            path = RESULTS / spec["tag"].format(seed=seed) / "grpo_logprob_preds.jsonl"
            rows = read_jsonl(path, 1884)

            seed_values.append([subject_accuracy(rows, subject) * 100
                                for subject in SUBJECT_ORDER])
        matrix[:, column] = np.mean(seed_values, axis=0) - 25
    return matrix

qwen_probe_landscape = endpoint_subject_probe(QWEN_PIPELINES)
gpt2_probe_landscape = endpoint_subject_probe(GPT2_PROBE_PIPELINES)
heat_limit = np.ceil(max(np.abs(qwen_probe_landscape).max(),
                         np.abs(gpt2_probe_landscape).max()) / 5) * 5

fig, axes = plt.subplots(1, 2, figsize=(12.8, 6.0), sharey=True, layout="constrained")
for ax, values, scales, title in (
    (axes[0], qwen_probe_landscape, list(QWEN_PIPELINES), "Qwen2.5"),
    (axes[1], gpt2_probe_landscape, list(GPT2_PROBE_PIPELINES), "GPT-2"),
):
    image = ax.imshow(values, cmap="RdYlGn", vmin=-heat_limit, vmax=heat_limit,
                      aspect="auto")
    for row in range(values.shape[0]):
        for column in range(values.shape[1]):
            value = values[row, column]
            text_colour = "white" if abs(value) > heat_limit * .58 else "#222222"
            ax.text(column, row, f"{value:+.1f}", ha="center", va="center",
                    fontsize=8, color=text_colour)
    for boundary in (2.5, 5.5):
        ax.axhline(boundary, color="white", linewidth=2)
    ax.set_xticks(range(len(scales)), scales)
    ax.set_xlabel("Within-family model scale")
    ax.set_title(title)
axes[0].set_yticks(range(len(SUBJECT_LABELS)), SUBJECT_LABELS)
fig.colorbar(image, ax=axes.ravel().tolist(), shrink=.82, pad=.03,
             label="Probe accuracy minus 25%\n(percentage points)")

finish_paper_plot(fig, "JF22", "C11_cross_family_subject_probe_landscape")

## Figure 15. Accuracy and format rewards across model families

In [ ]:
reward_composition_rows = []
for scale, spec in QWEN_PIPELINES.items():
    for seed in (0, 1, 2):
        path = RESULTS / spec["tag"].format(seed=seed) / "grpo_trainlog.json"
        history = read_json(path)
        rows = [row for row in history
                if "rewards/accuracy_reward/mean" in row
                and "rewards/format_reward/mean" in row]
        assert len(rows) == 500, f"{path}: incomplete reward history"

        reward_composition_rows.append({
            "family": "Qwen2.5", "scale": scale, "seed": seed,
            "accuracy": np.mean([row["rewards/accuracy_reward/mean"] for row in rows]),
            "format": np.mean([row["rewards/format_reward/mean"] for row in rows]),
        })
for row in gpt2_summary["training_summary"]:
    reward_composition_rows.append({
        "family": "GPT-2", "scale": row["scale"], "seed": int(row["seed"]),
        "accuracy": float(row["accuracy reward"]),
        "format": float(row["format reward"]),
    })
reward_composition_df = pd.DataFrame(reward_composition_rows)
family_scale_order = {
    "Qwen2.5": list(QWEN_PIPELINES),
    "GPT-2": list(GPT2_PROBE_PIPELINES),
}

fig, axes = plt.subplots(1, 2, figsize=(12.8, 5.2), sharey=True)
for ax, family in zip(axes, ("Qwen2.5", "GPT-2")):
    scale_order = family_scale_order[family]
    frame = reward_composition_df[reward_composition_df.family == family]
    accuracy_means = []
    format_means = []
    total_ranges = []
    for scale in scale_order:
        rows = frame[frame.scale == scale].sort_values("seed")
        accuracy_means.append(rows.accuracy.mean())
        format_means.append(rows.format.mean())
        totals = rows.accuracy.to_numpy() + rows.format.to_numpy()
        total_ranges.append((totals.mean(), totals.min(), totals.max()))
    positions = np.arange(len(scale_order))
    accuracy_means = np.asarray(accuracy_means)
    format_means = np.asarray(format_means)
    ax.bar(positions, accuracy_means, width=.62, color=PALETTE["orange"],
           label="Accuracy reward")
    ax.bar(positions, format_means, width=.62, bottom=accuracy_means,
           color=PALETTE["green"], label="Format reward")
    for index, (accuracy_mean, format_mean, total_range) in enumerate(
            zip(accuracy_means, format_means, total_ranges)):
        total, low, high = total_range
        ax.errorbar(index, total, yerr=[[total - low], [high - total]], fmt="D",
                    color="#222222", capsize=4, markersize=4, zorder=4)
        format_share = format_mean / (accuracy_mean + format_mean)
        ax.text(index, high + .035, f"{format_share:.0%} format", ha="center",
                va="bottom", fontsize=8)
    ax.set_xticks(positions, scale_order)
    finish_axes(ax, xlabel="Within-family model scale")
    ax.set_title(family)
axes[0].set_ylabel("Mean reward contribution")
axes[0].legend(ncol=2, loc="upper left")
axes[0].set_ylim(0, 1.04)
fig.tight_layout()

finish_paper_plot(fig, "JF23", "JF23_cross_family_reward_composition")

## Figure S4. KL divergence for full-parameter and LoRA updates

In [ ]:
def kl_trajectory(path, from_wandb=False):
    rows = ([json.loads(line) for line in Path(path).open() if line.strip()]
            if from_wandb else read_json(path))
    rows = [row for row in rows if row.get("kl") is not None]
    assert rows, f"{path}: no KL history"
    return np.array([row["step"] for row in rows]), np.array([row["kl"] for row in rows], dtype=float)

kl_full_b3_path = RESULTS / "035_technical" / "035b_q3_b3_full_wandb_history.jsonl"
kl_full_b15_wandb_path = RESULTS / "035_technical" / "035b_q3_b15_full_wandb_history.jsonl"
kl_full_b15_path = RESULTS / "035_COLAB_FULL_B15_matched_full_grpo_seed0" / "grpo_trainlog.json"
kl_lora_b3_path = RESULTS / "035_COLAB_FULL_B3_matched_lora_grpo_seed0" / "grpo_trainlog.json"
kl_lora_b15_path = RESULTS / "035_COLAB_FULL_B15_matched_lora_grpo_seed0" / "grpo_trainlog.json"

b15_train_steps, b15_train_kl = kl_trajectory(kl_full_b15_path)
b15_wandb_steps, b15_wandb_kl = kl_trajectory(kl_full_b15_wandb_path, from_wandb=True)
assert np.array_equal(b15_train_steps, b15_wandb_steps)
assert np.array_equal(b15_train_kl, b15_wandb_kl)
b3_steps, b3_kl = kl_trajectory(kl_full_b3_path, from_wandb=True)

KL_ARMS = [
    ("1.5B full", (b15_train_steps, b15_train_kl), PALETTE["orange"], "-", 1.5, 3),
    ("3B full", (b3_steps, b3_kl), PALETTE["red"], "-", 2.6, 5),
    ("3B LoRA", kl_trajectory(kl_lora_b3_path), PALETTE["blue"], "--", 1.2, 2),
    ("1.5B LoRA", kl_trajectory(kl_lora_b15_path), PALETTE["green"], "--", 1.2, 2),
]

fig, axes = plt.subplots(1, 2, figsize=(13.2, 5.1), gridspec_kw={"width_ratios": [1.32, 1]})
for label, (steps, values), colour, style, width, order in KL_ARMS:
    axes[0].plot(steps, np.where(values <= 0, np.nan, values), color=colour, linestyle=style,
                 linewidth=width, label=label, zorder=order, alpha=.95)
axes[0].axvspan(0, 50, color=PALETTE["light_grey"], alpha=.5, zorder=0)
axes[0].axvline(50, color="#444444", linestyle=":", linewidth=1.2, zorder=1)
axes[0].annotate("3B full model stopped here by the\nregistered A1 truncation rule", (b3_steps[-1], b3_kl[-1]),
                 xytext=(150, 3.2), fontsize=8.5, color=PALETTE["red"],
                 arrowprops=dict(arrowstyle="->", color=PALETTE["red"], linewidth=1.1, shrinkB=0))
axes[0].set_yscale("log")
axes[0].set_xlim(0, 500)
axes[0].set_ylim(3e-4, 7)
axes[0].set_xlabel("GRPO update")
axes[0].set_ylabel("KL from the reference policy (log scale)")
axes[0].grid(color="#E8E8E8", linewidth=.7)
axes[0].set_title("KL during GRPO")
axes[0].legend(loc="lower right", fontsize=9, ncol=2)

first_fifty = {label: float(np.mean(values[steps <= 50]))
               for label, (steps, values), *_ in KL_ARMS}
kl_pairs = [("3B", first_fifty["3B full"], first_fifty["3B LoRA"]),
            ("1.5B", first_fifty["1.5B full"], first_fifty["1.5B LoRA"])]
positions = np.arange(len(kl_pairs))
width = .38
axes[1].bar(positions - width / 2, [pair[1] for pair in kl_pairs], width=width,
            color=PALETTE["red"], alpha=.9, label="Full model")
axes[1].bar(positions + width / 2, [pair[2] for pair in kl_pairs], width=width,
            color=PALETTE["blue"], alpha=.9, label="Rank-16 LoRA")
for index, (name, full_value, lora_value) in enumerate(kl_pairs):
    axes[1].text(index - width / 2, full_value * 1.35, f"{full_value:.3f}", ha="center", fontsize=9)
    axes[1].text(index + width / 2, lora_value * 1.35, f"{lora_value:.5f}", ha="center", fontsize=9)
    ratio_y = (3e-4 * full_value) ** .5
    axes[1].text(index - width / 2, ratio_y, f"{full_value / lora_value:.0f}x",
                 ha="center", va="center", fontsize=11, fontweight="bold",
                 color="white")
axes[1].set_yscale("log")
axes[1].set_xticks(positions, [pair[0] for pair in kl_pairs])
axes[1].set_ylim(3e-4, 3)
axes[1].set_ylabel("Mean KL over the first 50 updates (log scale)")
axes[1].set_xlabel("Matched pair")
axes[1].grid(axis="y", color="#E8E8E8", linewidth=.7)
axes[1].set_title("Matched recipe and seed-matched start")
axes[1].legend(loc="upper center", fontsize=9, ncol=2)
fig.tight_layout()

finish_paper_plot(fig, "JF24", "JF24_matched_kl_separation")